# Game Preprocessing

This notebook prepares and cleans 2021 through 2025 MLB game data for modeling. The workflow includes organizing at-bats, engineering key features, and ensuring the dataset is fully ready for downstream analysis.

Although the primary focus of the project is modeling 2025 performance, incorporating several prior seasons provides the historical depth needed for exploratory data analysis (EDA). These earlier seasons help identify underlying trends and inform the structure and priors of the hierarchical model.


In [1]:
from pathlib import Path
from functools import lru_cache
import pandas as pd
import sys

In [2]:
# Find the repo root by searching upwards for src/hrmodel
here = Path.cwd()
for p in (here, *here.parents):
    if (p / "src" / "hrmodel").exists():
        REPO = p
        break
else:
    raise RuntimeError("Could not find repo root containing src/hrmodel")

# Put src/ on sys.path (front so it wins)
sys.path.insert(0, str(REPO / "src"))


In [3]:
# Import from your package
from hrmodel import (
    add_batter_names,
    add_batter_full_name,
    #prepare_barrels
)

# PyBaseball functions for Statcast data
from pybaseball import statcast
from pybaseball import playerid_reverse_lookup

# Core Python and data libraries
import numpy as np
import pandas as pd
from datetime import datetime
import os

from IPython.display import display, HTML

### Reading in Data

In [4]:
data_dir = Path("data/raw_season_data")
season_files = {y: data_dir / f"season_{y}.csv" for y in range(2021, 2026)}

read_kwargs = {
    # "usecols": [...],                 # select only needed columns
    # "parse_dates": [...],             # e.g., ["game_date"]
    # "dtype": {"batter_id": "int32"},  # downcast numerics where safe
    "engine": "pyarrow",                # faster & lower memory if available
    # "dtype_backend": "pyarrow",       # pandas 2.1+: keeps Arrow dtypes
}

@lru_cache(maxsize=None)
def load_season(year: int) -> pd.DataFrame:
    df = pd.read_csv(season_files[year], **read_kwargs)
    return df

# usage
season_2021 = load_season(2021)
season_2022 = load_season(2022)
season_2023 = load_season(2023)
season_2024 = load_season(2024)
season_2025 = load_season(2025)

In [5]:
# show specific ones
display(HTML("<h4>Season 2021</h4>")); display(season_2021.head(5))
display(HTML("<h4>Season 2022</h4>")); display(season_2022.head(5))
display(HTML("<h4>Season 2023</h4>")); display(season_2023.head(5))
display(HTML("<h4>Season 2024</h4>")); display(season_2024.head(5))
display(HTML("<h4>Season 2025</h4>")); display(season_2025.head(5))


,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,player_name,batter,pitcher,events,description,...,batter_days_until_next_game,api_break_z_with_gravity,api_break_x_arm,api_break_x_batter_in,arm_angle,attack_angle,attack_direction,swing_path_tilt,intercept_ball_minus_batter_pos_x_inches,intercept_ball_minus_batter_pos_y_inches
0,FF,2021-10-03,92.3,1.40,6.80,"Smith, Will",596019,519293,field_out,hit_into_play,...,NaN,1.28,0.69,-0.69,47.4,NaN,NaN,NaN,NaN,NaN
1,SL,2021-10-03,80.6,1.60,6.64,"Smith, Will",596019,519293,None,foul,...,NaN,2.99,-0.77,0.77,44.3,NaN,NaN,NaN,NaN,NaN
2,CU,2021-10-03,75.5,1.46,6.88,"Smith, Will",596019,519293,None,foul,...,NaN,4.52,-0.65,0.65,51.7,NaN,NaN,NaN,NaN,NaN
3,CU,2021-10-03,75.0,1.53,6.83,"Smith, Will",596019,519293,None,ball,...,NaN,4.74,-0.69,0.69,49.5,NaN,NaN,NaN,NaN,NaN
4,FF,2021-10-03,91.2,1.49,6.66,"Smith, Will",607043,519293,field_out,hit_into_play,...,NaN,1.49,0.63,0.63,44.0,NaN,NaN,NaN,NaN,NaN


,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,player_name,batter,pitcher,events,description,...,batter_days_until_next_game,api_break_z_with_gravity,api_break_x_arm,api_break_x_batter_in,arm_angle,attack_angle,attack_direction,swing_path_tilt,intercept_ball_minus_batter_pos_x_inches,intercept_ball_minus_batter_pos_y_inches
0,CH,2022-10-05,80.8,-0.76,6.61,"Baker, Bryan",624415,641329,field_out,hit_into_play,...,0.0,2.68,1.34,-1.34,59.9,NaN,NaN,NaN,NaN,NaN
1,FF,2022-10-05,97.7,-0.58,6.60,"Baker, Bryan",643376,641329,strikeout,swinging_strike,...,0.0,0.81,0.17,0.17,53.6,NaN,NaN,NaN,NaN,NaN
2,CH,2022-10-05,84.9,-0.55,6.58,"Baker, Bryan",643376,641329,None,ball,...,0.0,2.34,1.22,1.22,58.4,NaN,NaN,NaN,NaN,NaN
3,FF,2022-10-05,97.2,-0.42,6.60,"Baker, Bryan",643376,641329,None,swinging_strike,...,0.0,0.68,0.13,0.13,57.2,NaN,NaN,NaN,NaN,NaN
4,SL,2022-10-05,86.2,-0.55,6.64,"Baker, Bryan",643376,641329,None,called_strike,...,0.0,3.04,-0.63,-0.63,58.8,NaN,NaN,NaN,NaN,NaN


,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,player_name,batter,pitcher,events,description,...,batter_days_until_next_game,api_break_z_with_gravity,api_break_x_arm,api_break_x_batter_in,arm_angle,attack_angle,attack_direction,swing_path_tilt,intercept_ball_minus_batter_pos_x_inches,intercept_ball_minus_batter_pos_y_inches
0,CH,2023-10-01,89.0,-2.80,5.59,"Robertson, Nick",677008,687798,field_out,hit_into_play,...,NaN,2.55,1.53,-1.53,31.7,1.676715,-1.896554,41.830979,30.714944,26.412020
1,FF,2023-10-01,96.9,-2.40,5.90,"Robertson, Nick",677008,687798,None,foul,...,NaN,1.09,0.76,-0.76,47.4,8.715532,3.692542,40.551342,33.656454,26.020583
2,CH,2023-10-01,90.0,-2.93,5.56,"Robertson, Nick",677008,687798,None,ball,...,NaN,2.47,1.65,-1.65,30.3,NaN,NaN,NaN,NaN,NaN
3,ST,2023-10-01,82.2,-3.09,5.55,"Robertson, Nick",677008,687798,None,ball,...,NaN,3.14,-1.43,1.43,28.9,NaN,NaN,NaN,NaN,NaN
4,CH,2023-10-01,89.2,-2.87,5.58,"Robertson, Nick",677008,687798,None,swinging_strike,...,NaN,2.57,1.49,-1.49,34.3,20.169759,-7.584644,37.675911,44.236969,36.187039


,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,player_name,batter,pitcher,events,description,...,batter_days_until_next_game,api_break_z_with_gravity,api_break_x_arm,api_break_x_batter_in,arm_angle,attack_angle,attack_direction,swing_path_tilt,intercept_ball_minus_batter_pos_x_inches,intercept_ball_minus_batter_pos_y_inches
0,FF,2024-09-30,97.4,-2.10,4.88,"Díaz, Edwin",518595,621242,field_out,hit_into_play,...,1.0,1.40,0.96,0.96,17.6,6.149605,12.090516,22.160400,45.805662,22.048373
1,SL,2024-09-30,90.7,-2.14,5.06,"Díaz, Edwin",518595,621242,None,ball,...,1.0,2.14,-0.20,-0.20,23.1,NaN,NaN,NaN,NaN,NaN
2,SL,2024-09-30,91.1,-2.07,5.14,"Díaz, Edwin",518595,621242,None,swinging_strike,...,1.0,2.37,-0.12,-0.12,22.4,23.541699,-27.093819,34.778701,45.227965,45.368412
3,SL,2024-09-30,91.3,-2.05,5.07,"Díaz, Edwin",518595,621242,None,ball,...,1.0,2.09,-0.21,-0.21,22.4,NaN,NaN,NaN,NaN,NaN
4,SL,2024-09-30,89.1,-2.13,5.15,"Díaz, Edwin",518595,621242,None,swinging_strike,...,1.0,2.20,-0.17,-0.17,20.2,23.112048,-30.629825,33.038132,53.011806,51.686541


,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,player_name,batter,pitcher,events,description,...,batter_days_until_next_game,api_break_z_with_gravity,api_break_x_arm,api_break_x_batter_in,arm_angle,attack_angle,attack_direction,swing_path_tilt,intercept_ball_minus_batter_pos_x_inches,intercept_ball_minus_batter_pos_y_inches
0,FF,2025-09-28,95.7,-2.15,5.21,"Weissert, Greg",678009,669711,field_out,hit_into_play,...,2.0,1.56,0.71,-0.71,20.9,5.991833,-1.319512,28.782516,41.559201,30.599805
1,FF,2025-09-28,95.1,-1.91,5.10,"Weissert, Greg",668670,669711,strikeout,called_strike,...,9.0,1.59,0.93,0.93,20.5,NaN,NaN,NaN,NaN,NaN
2,FF,2025-09-28,95.4,-1.99,5.22,"Weissert, Greg",668670,669711,None,foul,...,9.0,1.36,0.85,0.85,22.9,2.871131,31.805044,22.266527,37.478847,15.582717
3,SL,2025-09-28,84.8,-2.33,4.72,"Weissert, Greg",668670,669711,None,swinging_strike,...,9.0,2.55,-0.32,-0.32,12.3,13.785410,4.081390,32.414181,38.011685,27.083341
4,SL,2025-09-28,85.3,-2.26,4.85,"Weissert, Greg",668670,669711,None,called_strike,...,9.0,2.71,-0.52,-0.52,15.8,NaN,NaN,NaN,NaN,NaN


## Examining Data

Below, I will examine the data dimension.

In [6]:
for y in range(2021, 2026):
    r, c = globals()[f"season_{y}"].shape
    print(f"season_{y}: {r:,} rows × {c} cols")

season_2021: 712,320 rows × 118 cols
season_2022: 710,210 rows × 118 cols
season_2023: 720,684 rows × 118 cols
season_2024: 732,481 rows × 118 cols
season_2025: 742,080 rows × 118 cols


### Column Consistency Check Across Seasons

- Defines the years to check (2021–2025).
- Uses `season_2025` as the reference column schema.
- Compares each `season_YYYY` dataset to the reference.
- Prints **OK** if column names *and order* match exactly, otherwise **DIFF**.
- Tracks whether all datasets match.
- Outputs a final `True/False` summary indicating full column consistency.

**Note:** This is a strict check — column order must also match.


In [7]:
years = range(2021, 2026)
ref = globals()["season_2025"].columns  # use 2025 as reference

print(f"[REFERENCE] season_2025 ({len(ref)} columns)\n")

all_match = True
for y in years:
    cols = globals()[f"season_{y}"].columns
    ok = cols.equals(ref)
    print(f"season_{y}: {'OK' if ok else 'DIFF'}")
    all_match &= ok

print("\nALL MATCH (names + order):", all_match)


[REFERENCE] season_2025 (118 columns)

season_2021: OK
season_2022: OK
season_2023: OK
season_2024: OK
season_2025: OK

ALL MATCH (names + order): True


### Printing Columns 

Since all columns match, I will print the `season_2025` columns.

In [8]:
season_2025.columns.tolist()

['pitch_type',
 'game_date',
 'release_speed',
 'release_pos_x',
 'release_pos_z',
 'player_name',
 'batter',
 'pitcher',
 'events',
 'description',
 'spin_dir',
 'spin_rate_deprecated',
 'break_angle_deprecated',
 'break_length_deprecated',
 'zone',
 'des',
 'game_type',
 'stand',
 'p_throws',
 'home_team',
 'away_team',
 'type',
 'hit_location',
 'bb_type',
 'balls',
 'strikes',
 'game_year',
 'pfx_x',
 'pfx_z',
 'plate_x',
 'plate_z',
 'on_3b',
 'on_2b',
 'on_1b',
 'outs_when_up',
 'inning',
 'inning_topbot',
 'hc_x',
 'hc_y',
 'tfs_deprecated',
 'tfs_zulu_deprecated',
 'umpire',
 'sv_id',
 'vx0',
 'vy0',
 'vz0',
 'ax',
 'ay',
 'az',
 'sz_top',
 'sz_bot',
 'hit_distance_sc',
 'launch_speed',
 'launch_angle',
 'effective_speed',
 'release_spin_rate',
 'release_extension',
 'game_pk',
 'fielder_2',
 'fielder_3',
 'fielder_4',
 'fielder_5',
 'fielder_6',
 'fielder_7',
 'fielder_8',
 'fielder_9',
 'release_pos_y',
 'estimated_ba_using_speedangle',
 'estimated_woba_using_speedangle',
 'w

### Verifying Home Runs Are Present in Every Season (Target Check)

This code loops through each season from 2021 to 2025 and confirms that the corresponding `season_YYYY` DataFrame is loaded and contains the `events` column. If a DataFrame is missing or the `events` column is not present, it prints a message and skips that season.

For each valid season, it checks whether the value `"home_run"` appears anywhere in `events` and also counts how many rows match `"home_run"`. The output reports **FOUND / NOT FOUND** along with the number of matching rows, which is especially important here because **home runs are the target variable**—it is vital that `"home_run"` exists in every season’s dataset (and ideally in nonzero quantity) to support training and evaluation.


In [9]:
target = "home_run"

for y in range(2021, 2026):
    name = f"season_{y}"
    df = globals().get(name)
    if df is None:
        print(f"{name}: (not loaded)")
        continue
    if "events" not in df.columns:
        print(f"{name}: 'events' column missing")
        continue

    has_it = df["events"].eq(target).any()
    count  = (df["events"] == target).sum()
    print(f"{name}: {'FOUND' if has_it else 'NOT FOUND'} — {count:,} rows")

season_2021: FOUND — 5,944 rows
season_2022: FOUND — 5,215 rows
season_2023: FOUND — 5,868 rows
season_2024: FOUND — 5,647 rows
season_2025: FOUND — 5,867 rows


### Inspecting `launch_speed_angle` Values Across Seasons

This code loops through each season from 2021 to 2025 and checks whether the corresponding `season_YYYY` DataFrame is loaded and contains the `launch_speed_angle` column. If a DataFrame is missing or the column does not exist, the code prints a clear message and skips that season.

For seasons where the column is present, the code extracts all unique values of `launch_speed_angle` and reports how many distinct values appear in that year. This serves as a quick consistency and sanity check to verify that the variable exists across seasons and to observe how its values vary over time. **Note that a value of `6` corresponds to a barrel.**


In [10]:
for y in range(2021, 2026):
    name = f"season_{y}"
    df = globals().get(name)
    if df is None:
        print(f"{name}: (not loaded)")
        continue
    if "launch_speed_angle" not in df.columns:
        print(f"{name}: (no 'launch_speed_angle' column)")
        continue

    uniques = df["launch_speed_angle"].unique()
    print(f"{name}: {len(uniques)} unique values")
    print(uniques, "\n")


season_2021: 7 unique values
[ 2. nan  3.  4.  6.  1.  5.] 

season_2022: 7 unique values
[ 3. nan  4.  6.  2.  5.  1.] 

season_2023: 7 unique values
[ 2. nan  3.  6.  4.  1.  5.] 

season_2024: 7 unique values
[ 2. nan  4.  3.  6.  1.  5.] 

season_2025: 7 unique values
[ 3. nan  2.  5.  4.  6.  1.] 



### Standardizing Column Names Across Seasons

This code standardizes column names across all seasonal datasets (2021–2025) by renaming player_name to pitcher_name and batter to batter_id, reporting for each season whether any renaming was applied or if the columns already matched (or were missing).

In [11]:
'''
Move to script
'''

def rename_cols(df, mapping):
    """Rename columns in-place; ignore missing; return list of actually changed names."""
    existing = {k: v for k, v in mapping.items() if k in df.columns and k != v}
    df.rename(columns=existing, inplace=True)
    return sorted(existing.items())  # list of (old, new) actually applied


In [12]:
rename_map = {
    "player_name": "pitcher_name",
    "batter": "batter_id",
}


# apply to season_2021 ... season_2025
for y in range(2021, 2026):
    name = f"season_{y}"
    df = globals().get(name)
    if df is None:
        print(f"{name}: (not loaded)")
        continue
    changed = rename_cols(df, rename_map)
    if changed:
        print(f"{name}: renamed -> {changed}")
    else:
        print(f"{name}: no changes (columns already matching or missing)")


season_2021: renamed -> [('batter', 'batter_id'), ('player_name', 'pitcher_name')]
season_2022: renamed -> [('batter', 'batter_id'), ('player_name', 'pitcher_name')]
season_2023: renamed -> [('batter', 'batter_id'), ('player_name', 'pitcher_name')]
season_2024: renamed -> [('batter', 'batter_id'), ('player_name', 'pitcher_name')]
season_2025: renamed -> [('batter', 'batter_id'), ('player_name', 'pitcher_name')]


### Adding Batter Name Columns Across Seasons

This code loops through each season from 2021 to 2025, checks that the corresponding `season_YYYY` DataFrame is loaded, and then applies two functions in sequence to enrich the dataset with batter name information. Because the Statcast data provides **player IDs rather than clean name fields**, these functions map batter IDs to player names and construct readable full-name columns. After each season’s DataFrame is updated, it is saved back into `globals()` so the changes persist for later steps in the workflow.

Finally, the code verifies that the expected columns—`batter_name` and `batter_full_name`—were successfully created and prints a clear status message for each season. This ensures that batter identities are consistently represented across all datasets before moving on to modeling or analysis.


In [13]:
for y in range(2021, 2026):
    name = f"season_{y}"
    df = globals().get(name)

    if df is None:
        print(f"{name}: (not loaded)")
        continue

    # Apply both functions sequentially and reassign each time
    df = add_batter_names(df)
    df = add_batter_full_name(df)
    globals()[name] = df

    # Verify both columns exist
    cols = df.columns
    has_batter_name = "batter_name" in cols
    has_batter_full_name = "batter_full_name" in cols

    # Construct a clean status message
    msg_parts = []
    msg_parts.append("✅ batter_name" if has_batter_name else "⚠️ missing batter_name")
    msg_parts.append("✅ batter_full_name" if has_batter_full_name else "⚠️ missing batter_full_name")

    print(f"{name}: " + " | ".join(msg_parts))

Gathering player lookup table. This may take a moment.
season_2021: ✅ batter_name | ⚠️ missing batter_full_name
season_2022: ✅ batter_name | ⚠️ missing batter_full_name
season_2023: ✅ batter_name | ⚠️ missing batter_full_name
season_2024: ✅ batter_name | ⚠️ missing batter_full_name
season_2025: ✅ batter_name | ⚠️ missing batter_full_name


In [14]:
# show specific ones
display(HTML("<h4>Season 2021</h4>")); display(season_2021.head(5))
display(HTML("<h4>Season 2022</h4>")); display(season_2022.head(5))
display(HTML("<h4>Season 2023</h4>")); display(season_2023.head(5))
display(HTML("<h4>Season 2024</h4>")); display(season_2024.head(5))
display(HTML("<h4>Season 2025</h4>")); display(season_2025.head(5))


,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,pitcher_name,batter_id,pitcher,events,description,...,intercept_ball_minus_batter_pos_y_inches,name_last,name_first,key_mlbam,key_retro,key_bbref,key_fangraphs,mlb_played_first,mlb_played_last,batter_name
0,FF,2021-10-03,92.3,1.40,6.80,"Smith, Will",596019,519293,field_out,hit_into_play,...,NaN,Lindor,Francisco,596019,lindf001,lindofr01,12916,2015.0,2025.0,Francisco Lindor
1,SL,2021-10-03,80.6,1.60,6.64,"Smith, Will",596019,519293,None,foul,...,NaN,Lindor,Francisco,596019,lindf001,lindofr01,12916,2015.0,2025.0,Francisco Lindor
2,CU,2021-10-03,75.5,1.46,6.88,"Smith, Will",596019,519293,None,foul,...,NaN,Lindor,Francisco,596019,lindf001,lindofr01,12916,2015.0,2025.0,Francisco Lindor
3,CU,2021-10-03,75.0,1.53,6.83,"Smith, Will",596019,519293,None,ball,...,NaN,Lindor,Francisco,596019,lindf001,lindofr01,12916,2015.0,2025.0,Francisco Lindor
4,FF,2021-10-03,91.2,1.49,6.66,"Smith, Will",607043,519293,field_out,hit_into_play,...,NaN,Nimmo,Brandon,607043,nimmb001,nimmobr01,12927,2016.0,2025.0,Brandon Nimmo


,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,pitcher_name,batter_id,pitcher,events,description,...,intercept_ball_minus_batter_pos_y_inches,name_last,name_first,key_mlbam,key_retro,key_bbref,key_fangraphs,mlb_played_first,mlb_played_last,batter_name
0,CH,2022-10-05,80.8,-0.76,6.61,"Baker, Bryan",624415,641329,field_out,hit_into_play,...,NaN,Biggio,Cavan,624415,biggc002,biggica01,19252,2019.0,2025.0,Cavan Biggio
1,FF,2022-10-05,97.7,-0.58,6.60,"Baker, Bryan",643376,641329,strikeout,swinging_strike,...,NaN,Jansen,Danny,643376,jansd001,janseda01,16535,2018.0,2025.0,Danny Jansen
2,CH,2022-10-05,84.9,-0.55,6.58,"Baker, Bryan",643376,641329,None,ball,...,NaN,Jansen,Danny,643376,jansd001,janseda01,16535,2018.0,2025.0,Danny Jansen
3,FF,2022-10-05,97.2,-0.42,6.60,"Baker, Bryan",643376,641329,None,swinging_strike,...,NaN,Jansen,Danny,643376,jansd001,janseda01,16535,2018.0,2025.0,Danny Jansen
4,SL,2022-10-05,86.2,-0.55,6.64,"Baker, Bryan",643376,641329,None,called_strike,...,NaN,Jansen,Danny,643376,jansd001,janseda01,16535,2018.0,2025.0,Danny Jansen


,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,pitcher_name,batter_id,pitcher,events,description,...,intercept_ball_minus_batter_pos_y_inches,name_last,name_first,key_mlbam,key_retro,key_bbref,key_fangraphs,mlb_played_first,mlb_played_last,batter_name
0,CH,2023-10-01,89.0,-2.80,5.59,"Robertson, Nick",677008,687798,field_out,hit_into_play,...,26.412020,Kjerstad,Heston,677008,kjerh001,kjershe01,31166,2023.0,2025.0,Heston Kjerstad
1,FF,2023-10-01,96.9,-2.40,5.90,"Robertson, Nick",677008,687798,None,foul,...,26.020583,Kjerstad,Heston,677008,kjerh001,kjershe01,31166,2023.0,2025.0,Heston Kjerstad
2,CH,2023-10-01,90.0,-2.93,5.56,"Robertson, Nick",677008,687798,None,ball,...,NaN,Kjerstad,Heston,677008,kjerh001,kjershe01,31166,2023.0,2025.0,Heston Kjerstad
3,ST,2023-10-01,82.2,-3.09,5.55,"Robertson, Nick",677008,687798,None,ball,...,NaN,Kjerstad,Heston,677008,kjerh001,kjershe01,31166,2023.0,2025.0,Heston Kjerstad
4,CH,2023-10-01,89.2,-2.87,5.58,"Robertson, Nick",677008,687798,None,swinging_strike,...,36.187039,Kjerstad,Heston,677008,kjerh001,kjershe01,31166,2023.0,2025.0,Heston Kjerstad


,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,pitcher_name,batter_id,pitcher,events,description,...,intercept_ball_minus_batter_pos_y_inches,name_last,name_first,key_mlbam,key_retro,key_bbref,key_fangraphs,mlb_played_first,mlb_played_last,batter_name
0,FF,2024-09-30,97.4,-2.10,4.88,"Díaz, Edwin",518595,621242,field_out,hit_into_play,...,22.048373,D'arnaud,Travis,518595.0,darnt001,darnatr01,7739.0,2013.0,2025.0,Travis D'arnaud
1,SL,2024-09-30,90.7,-2.14,5.06,"Díaz, Edwin",518595,621242,None,ball,...,NaN,D'arnaud,Travis,518595.0,darnt001,darnatr01,7739.0,2013.0,2025.0,Travis D'arnaud
2,SL,2024-09-30,91.1,-2.07,5.14,"Díaz, Edwin",518595,621242,None,swinging_strike,...,45.368412,D'arnaud,Travis,518595.0,darnt001,darnatr01,7739.0,2013.0,2025.0,Travis D'arnaud
3,SL,2024-09-30,91.3,-2.05,5.07,"Díaz, Edwin",518595,621242,None,ball,...,NaN,D'arnaud,Travis,518595.0,darnt001,darnatr01,7739.0,2013.0,2025.0,Travis D'arnaud
4,SL,2024-09-30,89.1,-2.13,5.15,"Díaz, Edwin",518595,621242,None,swinging_strike,...,51.686541,D'arnaud,Travis,518595.0,darnt001,darnatr01,7739.0,2013.0,2025.0,Travis D'arnaud


,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,pitcher_name,batter_id,pitcher,events,description,...,intercept_ball_minus_batter_pos_y_inches,name_last,name_first,key_mlbam,key_retro,key_bbref,key_fangraphs,mlb_played_first,mlb_played_last,batter_name
0,FF,2025-09-28,95.7,-2.15,5.21,"Weissert, Greg",678009,669711,field_out,hit_into_play,...,30.599805,Meadows,Parker,678009.0,meadp001,meadopa01,23800.0,2023.0,2025.0,Parker Meadows
1,FF,2025-09-28,95.1,-1.91,5.10,"Weissert, Greg",668670,669711,strikeout,called_strike,...,NaN,Rogers,Jake,668670.0,rogej004,rogerja03,19452.0,2019.0,2025.0,Jake Rogers
2,FF,2025-09-28,95.4,-1.99,5.22,"Weissert, Greg",668670,669711,None,foul,...,15.582717,Rogers,Jake,668670.0,rogej004,rogerja03,19452.0,2019.0,2025.0,Jake Rogers
3,SL,2025-09-28,84.8,-2.33,4.72,"Weissert, Greg",668670,669711,None,swinging_strike,...,27.083341,Rogers,Jake,668670.0,rogej004,rogerja03,19452.0,2019.0,2025.0,Jake Rogers
4,SL,2025-09-28,85.3,-2.26,4.85,"Weissert, Greg",668670,669711,None,called_strike,...,NaN,Rogers,Jake,668670.0,rogej004,rogerja03,19452.0,2019.0,2025.0,Jake Rogers


### Keeping Only Necessary Columns

To simplify the dataset and focus on the most relevant features for analysis, this step selects a subset of columns from the full `season_2025` DataFrame. These columns include game context (e.g., inning, teams), player identifiers, and pitch outcome details.

In [15]:
cols_to_keep = [
    'game_date',
    'pitcher_name',
    'home_team',
    'away_team',
    'inning',
    'inning_topbot',
    'at_bat_number',
    'batter_name',
    'batter_id',
    'pitch_number',
    'outs_when_up',
    'p_throws',
    'events',
    'description',
    'launch_speed_angle'
]

for y in range(2021, 2026):
    name = f"season_{y}"
    df = globals().get(name)
    if df is None:
        print(f"{name}: (not loaded)")
        continue

    # Keep only relevant columns (skip missing ones safely)
    available = [c for c in cols_to_keep if c in df.columns]
    globals()[name] = df[available]

    print(f"{name}: kept {len(available)} columns")


season_2021: kept 15 columns
season_2022: kept 15 columns
season_2023: kept 15 columns
season_2024: kept 15 columns
season_2025: kept 15 columns


## Barrels

Barrels are defined here as cases where `launch_speed_angle` equals `6`. Before calculating barrels, the data are filtered to retain only pitches that result in a recorded plate appearance outcome, as indicated by a non-empty `events` field. Because the outcome of an at-bat (e.g., a home run) is determined on the final pitch, this step effectively restricts the dataset to the **last pitch of each at-bat**, making the at-bat the unit of observation.

After filtering, a binary indicator is created to capture whether the final pitch of the at-bat was a barrel. This produces a cleaned, at-bat–level dataset for each season (stored as `at_bats_YYYY`) that is suitable for downstream analysis of barrel rates and home run outcomes.


In [16]:
'''
MOVE TO SCRIPT!
'''

def prepare_barrels(df: pd.DataFrame, barrel_col: str = "launch_speed_angle") -> pd.DataFrame:
    """
    Keep only rows with a non-empty `events` (i.e., PA result) and add `barrel` (==1 if barrel_col==6).
    """
    out = df.loc[df["events"].astype(str).str.strip().ne("") & df["events"].notna()].copy()
    out["barrel"] = (out[barrel_col] == 6).fillna(False).astype(int)
    return out


In [17]:
for y in range(2021, 2026):
    season_name = f"season_{y}"
    at_name     = f"at_bats_{y}"
    df = globals().get(season_name)
    if df is None:
        print(f"{season_name}: (not loaded)")
        continue
    globals()[at_name] = prepare_barrels(df)
    print(f"{at_name}: created (kept only rows with events)")


at_bats_2021: created (kept only rows with events)
at_bats_2022: created (kept only rows with events)
at_bats_2023: created (kept only rows with events)
at_bats_2024: created (kept only rows with events)
at_bats_2025: created (kept only rows with events)


### Double Checking Home Run Counts and Dimensions

To ensure that no home run observations were inadvertently removed during processing, I print the total number of home run events in each dataset and verify that these counts match the previously observed totals from earlier steps.

Also, in the cell below, I print out the dimensions of each season, confirming that the majority of rows were deleted as they were not the last at-bats.

In [18]:
print((at_bats_2021["events"] == "home_run").sum())
print((at_bats_2022["events"] == "home_run").sum())
print((at_bats_2023["events"] == "home_run").sum())
print((at_bats_2024["events"] == "home_run").sum())
print((at_bats_2025["events"] == "home_run").sum())

5944
5215
5868
5647
5867


In [19]:
for y in range(2021, 2026):
    r, c = globals()[f"at_bats_{y}"].shape
    print(f"season_{y}: {r:,} rows × {c} cols")

season_2021: 182,051 rows × 16 cols
season_2022: 182,349 rows × 16 cols
season_2023: 184,376 rows × 16 cols
season_2024: 189,416 rows × 16 cols
season_2025: 190,869 rows × 16 cols


In [20]:
# show specific ones
display(HTML("<h4>Season 2021</h4>")); display(at_bats_2021.head(5))
display(HTML("<h4>Season 2022</h4>")); display(at_bats_2022.head(5))
display(HTML("<h4>Season 2023</h4>")); display(at_bats_2023.head(5))
display(HTML("<h4>Season 2024</h4>")); display(at_bats_2024.head(5))
display(HTML("<h4>Season 2025</h4>")); display(at_bats_2025.head(5))


,game_date,pitcher_name,home_team,away_team,inning,inning_topbot,at_bat_number,batter_name,batter_id,pitch_number,outs_when_up,p_throws,events,description,launch_speed_angle,barrel
0,2021-10-03,"Smith, Will",ATL,NYM,9,Top,61,Francisco Lindor,596019,4,2,L,field_out,hit_into_play,2.0,0
4,2021-10-03,"Smith, Will",ATL,NYM,9,Top,60,Brandon Nimmo,607043,2,1,L,field_out,hit_into_play,2.0,0
6,2021-10-03,"Smith, Will",ATL,NYM,9,Top,59,Luis Guillorme,641645,3,0,L,strikeout,swinging_strike_blocked,NaN,0
9,2021-10-03,"Gsellman, Robert",ATL,NYM,8,Bot,58,Adam Duvall,594807,3,2,R,field_out,hit_into_play,3.0,0
12,2021-10-03,"Gsellman, Robert",ATL,NYM,8,Bot,57,Austin Riley,663586,6,1,R,strikeout,swinging_strike,NaN,0


,game_date,pitcher_name,home_team,away_team,inning,inning_topbot,at_bat_number,batter_name,batter_id,pitch_number,outs_when_up,p_throws,events,description,launch_speed_angle,barrel
0,2022-10-05,"Baker, Bryan",BAL,TOR,9,Top,78,Cavan Biggio,624415,1,2,R,field_out,hit_into_play,3.0,0
1,2022-10-05,"Baker, Bryan",BAL,TOR,9,Top,77,Danny Jansen,643376,5,1,R,strikeout,swinging_strike,NaN,0
6,2022-10-05,"Baker, Bryan",BAL,TOR,9,Top,76,Vladimir Guerrero,665489,4,0,R,strikeout,called_strike,NaN,0
10,2022-10-05,"White, Mitch",BAL,TOR,8,Bot,75,Jorge Mateo,622761,5,2,R,truncated_pa,ball,NaN,0
15,2022-10-05,"White, Mitch",BAL,TOR,8,Bot,74,Ryan Mckenna,663630,5,2,R,single,hit_into_play,4.0,0


,game_date,pitcher_name,home_team,away_team,inning,inning_topbot,at_bat_number,batter_name,batter_id,pitch_number,outs_when_up,p_throws,events,description,launch_speed_angle,barrel
0,2023-10-01,"Robertson, Nick",BAL,BOS,9,Bot,73,Heston Kjerstad,677008,6,2,R,field_out,hit_into_play,2.0,0
6,2023-10-01,"Robertson, Nick",BAL,BOS,9,Bot,72,Ramón Urías,602104,6,1,R,strikeout,swinging_strike,NaN,0
12,2023-10-01,"Robertson, Nick",BAL,BOS,9,Bot,71,Ryan Mountcastle,663624,5,0,R,field_out,hit_into_play,3.0,0
17,2023-10-01,"Irvin, Cole",BAL,BOS,9,Top,70,Ceddanne Rafaela,678882,5,2,L,strikeout,called_strike,NaN,0
22,2023-10-01,"Irvin, Cole",BAL,BOS,9,Top,69,Reese Mcguire,624512,4,1,L,field_out,hit_into_play,2.0,0


,game_date,pitcher_name,home_team,away_team,inning,inning_topbot,at_bat_number,batter_name,batter_id,pitch_number,outs_when_up,p_throws,events,description,launch_speed_angle,barrel
0,2024-09-30,"Díaz, Edwin",ATL,NYM,9,Bot,82,Travis D'arnaud,518595,5,2,R,field_out,hit_into_play,2.0,0
5,2024-09-30,"Díaz, Edwin",ATL,NYM,9,Bot,81,Ramón Laureano,657656,6,1,R,strikeout,swinging_strike,NaN,0
11,2024-09-30,"Díaz, Edwin",ATL,NYM,9,Bot,80,Eli White,642201,7,1,R,single,hit_into_play,4.0,0
18,2024-09-30,"Díaz, Edwin",ATL,NYM,9,Bot,79,Matt Olson,621566,1,0,R,field_out,hit_into_play,3.0,0
19,2024-09-30,"Johnson, Pierce",ATL,NYM,9,Top,78,Eddy Alvarez,657193,10,2,R,strikeout,swinging_strike_blocked,NaN,0


,game_date,pitcher_name,home_team,away_team,inning,inning_topbot,at_bat_number,batter_name,batter_id,pitch_number,outs_when_up,p_throws,events,description,launch_speed_angle,barrel
0,2025-09-28,"Weissert, Greg",BOS,DET,9,Top,74,Parker Meadows,678009,1,2,R,field_out,hit_into_play,3.0,0
1,2025-09-28,"Weissert, Greg",BOS,DET,9,Top,73,Jake Rogers,668670,4,1,R,strikeout,called_strike,NaN,0
5,2025-09-28,"Weissert, Greg",BOS,DET,9,Top,72,Andy Ibáñez,628451,7,1,R,single,hit_into_play,2.0,0
12,2025-09-28,"Weissert, Greg",BOS,DET,9,Top,71,Trey Sweeney,700242,6,1,R,walk,ball,NaN,0
18,2025-09-28,"Weissert, Greg",BOS,DET,9,Top,70,Javier Báez,595879,3,0,R,strikeout,swinging_strike,NaN,0


### Barrel Rate for Batters

The code below computes each batter’s rolling barrel rate over their most recent **10, 20, and 40 at-bats**. Before calculating these rolling features, the dataset is sorted (when available) by batter and plate-appearance context (`game_date`, `inning`, `at_bat_number`) to ensure the rolling windows follow the correct chronological order.

For each window, the rolling average is calculated within each batter group (`batter_id`) and **excludes the current at-bat** by shifting the series. This prevents look-ahead bias and ensures the features reflect only a batter’s prior performance, providing short-, medium-, and longer-term form indicators for downstream modeling.


In [21]:
for y in range(2021, 2026):
    name = f"at_bats_{y}"
    df = globals().get(name)
    if df is None:
        print(f"{name}: (not loaded)")
        continue

    # Sort first to ensure proper PA/order context
    sort_cols = ["batter_name", "game_date", "inning", "at_bat_number"]
    available = [c for c in sort_cols if c in df.columns]

    if "game_date" in available and not pd.api.types.is_datetime64_any_dtype(df["game_date"]):
        df["game_date"] = pd.to_datetime(df["game_date"], errors="coerce")

    if available:
        df.sort_values(by=available, inplace=True, kind="mergesort")

    # Check required columns for rolling calc
    needed = ("batter_id", "barrel")
    missing = [c for c in needed if c not in df.columns]
    if missing:
        print(f"{name}: missing columns {missing}")
        continue

    # Compute rolling averages: 40, 20, 10 (shift(1) to exclude current AB)
    for window in [40, 20, 10]:
        col_name = f"rolling_batter_barrel_rate_{window}"
        df[col_name] = (
            df.groupby("batter_id")["barrel"]
              .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
        )

    print(f"{name}: sorted and added rolling barrel rates (10, 20, 40)")


at_bats_2021: sorted and added rolling barrel rates (10, 20, 40)
at_bats_2022: sorted and added rolling barrel rates (10, 20, 40)
at_bats_2023: sorted and added rolling barrel rates (10, 20, 40)
at_bats_2024: sorted and added rolling barrel rates (10, 20, 40)
at_bats_2025: sorted and added rolling barrel rates (10, 20, 40)


In [22]:
display(HTML("<h4>Season 2021</h4>")); display(at_bats_2021.head(5))
display(HTML("<h4>Season 2022</h4>")); display(at_bats_2022.head(5))
display(HTML("<h4>Season 2023</h4>")); display(at_bats_2023.head(5))
display(HTML("<h4>Season 2024</h4>")); display(at_bats_2024.head(5))
display(HTML("<h4>Season 2025</h4>")); display(at_bats_2025.head(5))


,game_date,pitcher_name,home_team,away_team,inning,inning_topbot,at_bat_number,batter_name,batter_id,pitch_number,outs_when_up,p_throws,events,description,launch_speed_angle,barrel,rolling_batter_barrel_rate_40,rolling_batter_barrel_rate_20,rolling_batter_barrel_rate_10
5474,2021-10-02,"Gsellman, Robert",ATL,NYM,6,Bot,51,A. j. Minter,621345,5,1,R,strikeout,missed_bunt,NaN,0,NaN,NaN,NaN
199785,2021-08-14,"Keller, Mitch",PIT,MIL,3,Top,19,Aaron Ashby,676879,4,0,R,strikeout,swinging_strike,NaN,0,NaN,NaN,NaN
199742,2021-08-14,"Keller, Mitch",PIT,MIL,4,Top,31,Aaron Ashby,676879,3,2,R,strikeout,swinging_strike,NaN,0,0.0,0.0,0.0
123141,2021-09-03,"Wainwright, Adam",MIL,STL,3,Bot,28,Aaron Ashby,676879,1,1,R,sac_bunt,hit_into_play,1.0,0,0.0,0.0,0.0
123071,2021-09-03,"Wainwright, Adam",MIL,STL,6,Bot,46,Aaron Ashby,676879,7,0,R,walk,ball,NaN,0,0.0,0.0,0.0


,game_date,pitcher_name,home_team,away_team,inning,inning_topbot,at_bat_number,batter_name,batter_id,pitch_number,outs_when_up,p_throws,events,description,launch_speed_angle,barrel,rolling_batter_barrel_rate_40,rolling_batter_barrel_rate_20,rolling_batter_barrel_rate_10
707511,2022-04-08,"Eovaldi, Nathan",NYY,BOS,2,Bot,17,Aaron Hicks,543305,8,1,R,strikeout,swinging_strike,NaN,0,NaN,NaN,NaN
707450,2022-04-08,"Eovaldi, Nathan",NYY,BOS,4,Bot,33,Aaron Hicks,543305,4,1,R,strikeout,swinging_strike,NaN,0,0.0,0.0,0.0
707385,2022-04-08,"Whitlock, Garrett",NYY,BOS,6,Bot,52,Aaron Hicks,543305,4,1,R,single,hit_into_play,4.0,0,0.0,0.0,0.0
707321,2022-04-08,"Strahm, Matt",NYY,BOS,8,Bot,67,Aaron Hicks,543305,6,1,L,strikeout,swinging_strike,NaN,0,0.0,0.0,0.0
707268,2022-04-08,"Diekman, Jake",NYY,BOS,10,Bot,83,Aaron Hicks,543305,4,1,L,intent_walk,automatic_ball,NaN,0,0.0,0.0,0.0


,game_date,pitcher_name,home_team,away_team,inning,inning_topbot,at_bat_number,batter_name,batter_id,pitch_number,outs_when_up,p_throws,events,description,launch_speed_angle,barrel,rolling_batter_barrel_rate_40,rolling_batter_barrel_rate_20,rolling_batter_barrel_rate_10
710237,2023-04-01,"Doval, Camilo",NYY,SF,9,Bot,79,Aaron Hicks,543305,7,0,R,strikeout,called_strike,NaN,0,NaN,NaN,NaN
703643,2023-04-03,"Walker, Taijuan",NYY,PHI,1,Bot,10,Aaron Hicks,543305,6,2,R,walk,ball,NaN,0,0.0,0.0,0.0
703557,2023-04-03,"Walker, Taijuan",NYY,PHI,4,Bot,33,Aaron Hicks,543305,4,0,R,field_out,hit_into_play,2.0,0,0.0,0.0,0.0
703504,2023-04-03,"Marte, Yunior",NYY,PHI,5,Bot,46,Aaron Hicks,543305,4,1,R,strikeout,swinging_strike,NaN,0,0.0,0.0,0.0
703438,2023-04-03,"Vasquez, Andrew",NYY,PHI,7,Bot,64,Aaron Hicks,543305,2,1,L,field_out,hit_into_play,2.0,0,0.0,0.0,0.0


,game_date,pitcher_name,home_team,away_team,inning,inning_topbot,at_bat_number,batter_name,batter_id,pitch_number,outs_when_up,p_throws,events,description,launch_speed_angle,barrel,rolling_batter_barrel_rate_40,rolling_batter_barrel_rate_20,rolling_batter_barrel_rate_10
731616,2024-03-20,"Hicks, Jordan",LAA,SF,1,Bot,6,Aaron Hicks,543305,1,0,R,single,hit_into_play,NaN,0,NaN,NaN,NaN
731587,2024-03-20,"Hicks, Jordan",LAA,SF,3,Bot,22,Aaron Hicks,543305,1,0,R,field_out,hit_into_play,NaN,0,0.0,0.0,0.0
731548,2024-03-20,"Frisbee, Matt",LAA,SF,5,Bot,40,Aaron Hicks,543305,3,2,R,strikeout,swinging_strike,NaN,0,0.0,0.0,0.0
726328,2024-03-22,"Cannon, Jonathan",LAA,CWS,1,Bot,5,Aaron Hicks,543305,1,1,R,single,hit_into_play,NaN,0,0.0,0.0,0.0
726299,2024-03-22,"Cannon, Jonathan",LAA,CWS,3,Bot,21,Aaron Hicks,543305,1,1,R,single,hit_into_play,NaN,0,0.0,0.0,0.0


,game_date,pitcher_name,home_team,away_team,inning,inning_topbot,at_bat_number,batter_name,batter_id,pitch_number,outs_when_up,p_throws,events,description,launch_speed_angle,barrel,rolling_batter_barrel_rate_40,rolling_batter_barrel_rate_20,rolling_batter_barrel_rate_10
739476,2025-03-18,"Buehler, Walker",NYY,BOS,1,Bot,6,Aaron Judge,592450,4,1,R,strikeout,called_strike,NaN,0,NaN,NaN,NaN
739398,2025-03-18,"Buehler, Walker",NYY,BOS,4,Bot,25,Aaron Judge,592450,3,1,R,double,hit_into_play,6.0,1,0.000000,0.000000,0.000000
739361,2025-03-18,"Buehler, Walker",NYY,BOS,5,Bot,37,Aaron Judge,592450,4,2,R,force_out,hit_into_play,2.0,0,0.500000,0.500000,0.500000
735907,2025-03-19,"Schwellenbach, Spencer",NYY,ATL,1,Bot,6,Aaron Judge,592450,4,1,R,strikeout,swinging_strike,NaN,0,0.333333,0.333333,0.333333
735839,2025-03-19,"Schwellenbach, Spencer",NYY,ATL,4,Bot,25,Aaron Judge,592450,3,0,R,force_out,hit_into_play,2.0,0,0.250000,0.250000,0.250000


### Barrel Rate for Pitchers

The code below computes rolling barrel rates **allowed by each pitcher** over their most recent **10, 20, and 40 at-bats**. For each window, the rolling average is calculated within pitcher groups and **explicitly excludes the current at-bat** by shifting the series, ensuring that only prior information is used.

These rolling barrel-rate measures capture short-, medium-, and longer-term pitcher tendencies and will serve as key predictors in the Bayesian model, allowing the model to incorporate recent form while avoiding look-ahead bias.


In [23]:
for y in range(2021, 2026):
    name = f"at_bats_{y}"
    df = globals().get(name)
    if df is None:
        print(f"{name}: (not loaded)")
        continue

    # Check required columns
    needed = ("pitcher_name", "barrel")
    missing = [c for c in needed if c not in df.columns]
    if missing:
        print(f"{name}: missing columns {missing}")
        continue

    # Compute 3 rolling averages: 40, 20, 10
    for window in [40, 20, 10]:
        col_name = f"rolling_pitcher_barrel_rate_{window}"
        df[col_name] = (
            df.groupby("pitcher_name")["barrel"]
              .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
        )

    print(f"{name}: added rolling pitcher barrel rates (10, 20, 40)")


at_bats_2021: added rolling pitcher barrel rates (10, 20, 40)
at_bats_2022: added rolling pitcher barrel rates (10, 20, 40)
at_bats_2023: added rolling pitcher barrel rates (10, 20, 40)
at_bats_2024: added rolling pitcher barrel rates (10, 20, 40)
at_bats_2025: added rolling pitcher barrel rates (10, 20, 40)


In [24]:
# show specific ones
display(HTML("<h4>Season 2021</h4>")); display(at_bats_2021.head(5))
display(HTML("<h4>Season 2022</h4>")); display(at_bats_2022.head(5))
display(HTML("<h4>Season 2023</h4>")); display(at_bats_2023.head(5))
display(HTML("<h4>Season 2024</h4>")); display(at_bats_2024.head(5))
display(HTML("<h4>Season 2025</h4>")); display(at_bats_2025.head(5))


,game_date,pitcher_name,home_team,away_team,inning,inning_topbot,at_bat_number,batter_name,batter_id,pitch_number,...,events,description,launch_speed_angle,barrel,rolling_batter_barrel_rate_40,rolling_batter_barrel_rate_20,rolling_batter_barrel_rate_10,rolling_pitcher_barrel_rate_40,rolling_pitcher_barrel_rate_20,rolling_pitcher_barrel_rate_10
5474,2021-10-02,"Gsellman, Robert",ATL,NYM,6,Bot,51,A. j. Minter,621345,5,...,strikeout,missed_bunt,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN
199785,2021-08-14,"Keller, Mitch",PIT,MIL,3,Top,19,Aaron Ashby,676879,4,...,strikeout,swinging_strike,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN
199742,2021-08-14,"Keller, Mitch",PIT,MIL,4,Top,31,Aaron Ashby,676879,3,...,strikeout,swinging_strike,NaN,0,0.0,0.0,0.0,0.0,0.0,0.0
123141,2021-09-03,"Wainwright, Adam",MIL,STL,3,Bot,28,Aaron Ashby,676879,1,...,sac_bunt,hit_into_play,1.0,0,0.0,0.0,0.0,NaN,NaN,NaN
123071,2021-09-03,"Wainwright, Adam",MIL,STL,6,Bot,46,Aaron Ashby,676879,7,...,walk,ball,NaN,0,0.0,0.0,0.0,0.0,0.0,0.0


,game_date,pitcher_name,home_team,away_team,inning,inning_topbot,at_bat_number,batter_name,batter_id,pitch_number,...,events,description,launch_speed_angle,barrel,rolling_batter_barrel_rate_40,rolling_batter_barrel_rate_20,rolling_batter_barrel_rate_10,rolling_pitcher_barrel_rate_40,rolling_pitcher_barrel_rate_20,rolling_pitcher_barrel_rate_10
707511,2022-04-08,"Eovaldi, Nathan",NYY,BOS,2,Bot,17,Aaron Hicks,543305,8,...,strikeout,swinging_strike,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN
707450,2022-04-08,"Eovaldi, Nathan",NYY,BOS,4,Bot,33,Aaron Hicks,543305,4,...,strikeout,swinging_strike,NaN,0,0.0,0.0,0.0,0.0,0.0,0.0
707385,2022-04-08,"Whitlock, Garrett",NYY,BOS,6,Bot,52,Aaron Hicks,543305,4,...,single,hit_into_play,4.0,0,0.0,0.0,0.0,NaN,NaN,NaN
707321,2022-04-08,"Strahm, Matt",NYY,BOS,8,Bot,67,Aaron Hicks,543305,6,...,strikeout,swinging_strike,NaN,0,0.0,0.0,0.0,NaN,NaN,NaN
707268,2022-04-08,"Diekman, Jake",NYY,BOS,10,Bot,83,Aaron Hicks,543305,4,...,intent_walk,automatic_ball,NaN,0,0.0,0.0,0.0,NaN,NaN,NaN


,game_date,pitcher_name,home_team,away_team,inning,inning_topbot,at_bat_number,batter_name,batter_id,pitch_number,...,events,description,launch_speed_angle,barrel,rolling_batter_barrel_rate_40,rolling_batter_barrel_rate_20,rolling_batter_barrel_rate_10,rolling_pitcher_barrel_rate_40,rolling_pitcher_barrel_rate_20,rolling_pitcher_barrel_rate_10
710237,2023-04-01,"Doval, Camilo",NYY,SF,9,Bot,79,Aaron Hicks,543305,7,...,strikeout,called_strike,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN
703643,2023-04-03,"Walker, Taijuan",NYY,PHI,1,Bot,10,Aaron Hicks,543305,6,...,walk,ball,NaN,0,0.0,0.0,0.0,NaN,NaN,NaN
703557,2023-04-03,"Walker, Taijuan",NYY,PHI,4,Bot,33,Aaron Hicks,543305,4,...,field_out,hit_into_play,2.0,0,0.0,0.0,0.0,0.0,0.0,0.0
703504,2023-04-03,"Marte, Yunior",NYY,PHI,5,Bot,46,Aaron Hicks,543305,4,...,strikeout,swinging_strike,NaN,0,0.0,0.0,0.0,NaN,NaN,NaN
703438,2023-04-03,"Vasquez, Andrew",NYY,PHI,7,Bot,64,Aaron Hicks,543305,2,...,field_out,hit_into_play,2.0,0,0.0,0.0,0.0,NaN,NaN,NaN


,game_date,pitcher_name,home_team,away_team,inning,inning_topbot,at_bat_number,batter_name,batter_id,pitch_number,...,events,description,launch_speed_angle,barrel,rolling_batter_barrel_rate_40,rolling_batter_barrel_rate_20,rolling_batter_barrel_rate_10,rolling_pitcher_barrel_rate_40,rolling_pitcher_barrel_rate_20,rolling_pitcher_barrel_rate_10
731616,2024-03-20,"Hicks, Jordan",LAA,SF,1,Bot,6,Aaron Hicks,543305,1,...,single,hit_into_play,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN
731587,2024-03-20,"Hicks, Jordan",LAA,SF,3,Bot,22,Aaron Hicks,543305,1,...,field_out,hit_into_play,NaN,0,0.0,0.0,0.0,0.0,0.0,0.0
731548,2024-03-20,"Frisbee, Matt",LAA,SF,5,Bot,40,Aaron Hicks,543305,3,...,strikeout,swinging_strike,NaN,0,0.0,0.0,0.0,NaN,NaN,NaN
726328,2024-03-22,"Cannon, Jonathan",LAA,CWS,1,Bot,5,Aaron Hicks,543305,1,...,single,hit_into_play,NaN,0,0.0,0.0,0.0,NaN,NaN,NaN
726299,2024-03-22,"Cannon, Jonathan",LAA,CWS,3,Bot,21,Aaron Hicks,543305,1,...,single,hit_into_play,NaN,0,0.0,0.0,0.0,0.0,0.0,0.0


,game_date,pitcher_name,home_team,away_team,inning,inning_topbot,at_bat_number,batter_name,batter_id,pitch_number,...,events,description,launch_speed_angle,barrel,rolling_batter_barrel_rate_40,rolling_batter_barrel_rate_20,rolling_batter_barrel_rate_10,rolling_pitcher_barrel_rate_40,rolling_pitcher_barrel_rate_20,rolling_pitcher_barrel_rate_10
739476,2025-03-18,"Buehler, Walker",NYY,BOS,1,Bot,6,Aaron Judge,592450,4,...,strikeout,called_strike,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN
739398,2025-03-18,"Buehler, Walker",NYY,BOS,4,Bot,25,Aaron Judge,592450,3,...,double,hit_into_play,6.0,1,0.000000,0.000000,0.000000,0.0,0.0,0.0
739361,2025-03-18,"Buehler, Walker",NYY,BOS,5,Bot,37,Aaron Judge,592450,4,...,force_out,hit_into_play,2.0,0,0.500000,0.500000,0.500000,0.5,0.5,0.5
735907,2025-03-19,"Schwellenbach, Spencer",NYY,ATL,1,Bot,6,Aaron Judge,592450,4,...,strikeout,swinging_strike,NaN,0,0.333333,0.333333,0.333333,NaN,NaN,NaN
735839,2025-03-19,"Schwellenbach, Spencer",NYY,ATL,4,Bot,25,Aaron Judge,592450,3,...,force_out,hit_into_play,2.0,0,0.250000,0.250000,0.250000,0.0,0.0,0.0


## Homerun


Now, we will create an indicator for the binary response variable, showing whether or not a player hit a home run on the final pitch of an at-bat.

This code iterates over the `at_bats_YYYY` datasets (2021–2025), sorts each one using available plate-appearance ordering columns to preserve the correct chronological context, and then creates a binary `home_run` indicator (`1` if `events == "home_run"`, otherwise `0`). This indicator serves as the binary response variable, showing whether or not a player hit a home run on the final pitch of an at-bat. If the dataset is not loaded or the `events` column is missing, the code prints a message and skips the feature creation for that season.


In [25]:
for y in range(2021, 2026):
    name = f"at_bats_{y}"
    df = globals().get(name)
    if df is None:
        print(f"{name}: (not loaded)")
        continue

    # Sort (use only columns that exist to avoid KeyError)
    sort_cols = ["batter_name", "game_date", "inning", "at_bat_number"]
    available = [c for c in sort_cols if c in df.columns]
    if available:
        df.sort_values(by=available, inplace=True, kind="mergesort")  # stable
    else:
        print(f"{name}: no sort performed (none of the sort columns found)")

    # Add binary home_run indicator
    if "events" in df.columns:
        df["home_run"] = (df["events"] == "home_run").astype(int)
        print(f"{name}: sorted and home_run column added")
    else:
        print(f"{name}: 'events' column missing; skipped home_run creation")


at_bats_2021: sorted and home_run column added
at_bats_2022: sorted and home_run column added
at_bats_2023: sorted and home_run column added
at_bats_2024: sorted and home_run column added
at_bats_2025: sorted and home_run column added


In [26]:
display(HTML("<h4>Season 2021</h4>")); display(at_bats_2021.head(5))
display(HTML("<h4>Season 2022</h4>")); display(at_bats_2022.head(5))
display(HTML("<h4>Season 2023</h4>")); display(at_bats_2023.head(5))
display(HTML("<h4>Season 2024</h4>")); display(at_bats_2024.head(5))
display(HTML("<h4>Season 2025</h4>")); display(at_bats_2025.head(5))


,game_date,pitcher_name,home_team,away_team,inning,inning_topbot,at_bat_number,batter_name,batter_id,pitch_number,...,description,launch_speed_angle,barrel,rolling_batter_barrel_rate_40,rolling_batter_barrel_rate_20,rolling_batter_barrel_rate_10,rolling_pitcher_barrel_rate_40,rolling_pitcher_barrel_rate_20,rolling_pitcher_barrel_rate_10,home_run
5474,2021-10-02,"Gsellman, Robert",ATL,NYM,6,Bot,51,A. j. Minter,621345,5,...,missed_bunt,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,0
199785,2021-08-14,"Keller, Mitch",PIT,MIL,3,Top,19,Aaron Ashby,676879,4,...,swinging_strike,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,0
199742,2021-08-14,"Keller, Mitch",PIT,MIL,4,Top,31,Aaron Ashby,676879,3,...,swinging_strike,NaN,0,0.0,0.0,0.0,0.0,0.0,0.0,0
123141,2021-09-03,"Wainwright, Adam",MIL,STL,3,Bot,28,Aaron Ashby,676879,1,...,hit_into_play,1.0,0,0.0,0.0,0.0,NaN,NaN,NaN,0
123071,2021-09-03,"Wainwright, Adam",MIL,STL,6,Bot,46,Aaron Ashby,676879,7,...,ball,NaN,0,0.0,0.0,0.0,0.0,0.0,0.0,0


,game_date,pitcher_name,home_team,away_team,inning,inning_topbot,at_bat_number,batter_name,batter_id,pitch_number,...,description,launch_speed_angle,barrel,rolling_batter_barrel_rate_40,rolling_batter_barrel_rate_20,rolling_batter_barrel_rate_10,rolling_pitcher_barrel_rate_40,rolling_pitcher_barrel_rate_20,rolling_pitcher_barrel_rate_10,home_run
707511,2022-04-08,"Eovaldi, Nathan",NYY,BOS,2,Bot,17,Aaron Hicks,543305,8,...,swinging_strike,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,0
707450,2022-04-08,"Eovaldi, Nathan",NYY,BOS,4,Bot,33,Aaron Hicks,543305,4,...,swinging_strike,NaN,0,0.0,0.0,0.0,0.0,0.0,0.0,0
707385,2022-04-08,"Whitlock, Garrett",NYY,BOS,6,Bot,52,Aaron Hicks,543305,4,...,hit_into_play,4.0,0,0.0,0.0,0.0,NaN,NaN,NaN,0
707321,2022-04-08,"Strahm, Matt",NYY,BOS,8,Bot,67,Aaron Hicks,543305,6,...,swinging_strike,NaN,0,0.0,0.0,0.0,NaN,NaN,NaN,0
707268,2022-04-08,"Diekman, Jake",NYY,BOS,10,Bot,83,Aaron Hicks,543305,4,...,automatic_ball,NaN,0,0.0,0.0,0.0,NaN,NaN,NaN,0


,game_date,pitcher_name,home_team,away_team,inning,inning_topbot,at_bat_number,batter_name,batter_id,pitch_number,...,description,launch_speed_angle,barrel,rolling_batter_barrel_rate_40,rolling_batter_barrel_rate_20,rolling_batter_barrel_rate_10,rolling_pitcher_barrel_rate_40,rolling_pitcher_barrel_rate_20,rolling_pitcher_barrel_rate_10,home_run
710237,2023-04-01,"Doval, Camilo",NYY,SF,9,Bot,79,Aaron Hicks,543305,7,...,called_strike,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,0
703643,2023-04-03,"Walker, Taijuan",NYY,PHI,1,Bot,10,Aaron Hicks,543305,6,...,ball,NaN,0,0.0,0.0,0.0,NaN,NaN,NaN,0
703557,2023-04-03,"Walker, Taijuan",NYY,PHI,4,Bot,33,Aaron Hicks,543305,4,...,hit_into_play,2.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0
703504,2023-04-03,"Marte, Yunior",NYY,PHI,5,Bot,46,Aaron Hicks,543305,4,...,swinging_strike,NaN,0,0.0,0.0,0.0,NaN,NaN,NaN,0
703438,2023-04-03,"Vasquez, Andrew",NYY,PHI,7,Bot,64,Aaron Hicks,543305,2,...,hit_into_play,2.0,0,0.0,0.0,0.0,NaN,NaN,NaN,0


,game_date,pitcher_name,home_team,away_team,inning,inning_topbot,at_bat_number,batter_name,batter_id,pitch_number,...,description,launch_speed_angle,barrel,rolling_batter_barrel_rate_40,rolling_batter_barrel_rate_20,rolling_batter_barrel_rate_10,rolling_pitcher_barrel_rate_40,rolling_pitcher_barrel_rate_20,rolling_pitcher_barrel_rate_10,home_run
731616,2024-03-20,"Hicks, Jordan",LAA,SF,1,Bot,6,Aaron Hicks,543305,1,...,hit_into_play,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,0
731587,2024-03-20,"Hicks, Jordan",LAA,SF,3,Bot,22,Aaron Hicks,543305,1,...,hit_into_play,NaN,0,0.0,0.0,0.0,0.0,0.0,0.0,0
731548,2024-03-20,"Frisbee, Matt",LAA,SF,5,Bot,40,Aaron Hicks,543305,3,...,swinging_strike,NaN,0,0.0,0.0,0.0,NaN,NaN,NaN,0
726328,2024-03-22,"Cannon, Jonathan",LAA,CWS,1,Bot,5,Aaron Hicks,543305,1,...,hit_into_play,NaN,0,0.0,0.0,0.0,NaN,NaN,NaN,0
726299,2024-03-22,"Cannon, Jonathan",LAA,CWS,3,Bot,21,Aaron Hicks,543305,1,...,hit_into_play,NaN,0,0.0,0.0,0.0,0.0,0.0,0.0,0


,game_date,pitcher_name,home_team,away_team,inning,inning_topbot,at_bat_number,batter_name,batter_id,pitch_number,...,description,launch_speed_angle,barrel,rolling_batter_barrel_rate_40,rolling_batter_barrel_rate_20,rolling_batter_barrel_rate_10,rolling_pitcher_barrel_rate_40,rolling_pitcher_barrel_rate_20,rolling_pitcher_barrel_rate_10,home_run
739476,2025-03-18,"Buehler, Walker",NYY,BOS,1,Bot,6,Aaron Judge,592450,4,...,called_strike,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,0
739398,2025-03-18,"Buehler, Walker",NYY,BOS,4,Bot,25,Aaron Judge,592450,3,...,hit_into_play,6.0,1,0.000000,0.000000,0.000000,0.0,0.0,0.0,0
739361,2025-03-18,"Buehler, Walker",NYY,BOS,5,Bot,37,Aaron Judge,592450,4,...,hit_into_play,2.0,0,0.500000,0.500000,0.500000,0.5,0.5,0.5,0
735907,2025-03-19,"Schwellenbach, Spencer",NYY,ATL,1,Bot,6,Aaron Judge,592450,4,...,swinging_strike,NaN,0,0.333333,0.333333,0.333333,NaN,NaN,NaN,0
735839,2025-03-19,"Schwellenbach, Spencer",NYY,ATL,4,Bot,25,Aaron Judge,592450,3,...,hit_into_play,2.0,0,0.250000,0.250000,0.250000,0.0,0.0,0.0,0


### Homerun Previous Game

Another set of predictors captures whether a batter hit a home run in their **most recent games** to help represent potential “hot streak” effects, where recent success may increase the likelihood of another home run. The code first aggregates the data to the **batter–game level** (summing home runs within each game and converting this to a binary `hit_hr` flag), orders games chronologically by `game_date`, and then computes lagged indicators that **exclude the current game**.

Specifically, it creates binary features for whether the batter homered in the **previous game** (`prev_game_hr`), in **either of the previous 2 games** (`prev_2g_hr`), or in **either of the previous 5 games** (`prev_5g_hr`). These game-level features are then merged back onto the at-bat data so every at-bat in a given game carries the same recent-home-run indicators.


In [27]:
for y in range(2021, 2026):
    name = f"at_bats_{y}"
    df = globals().get(name)
    if df is None:
        print(f"{name}: (not loaded)")
        continue

    need = {"batter_name", "game_date", "home_run"}
    if not need.issubset(df.columns):
        print(f"{name}: missing columns {sorted(need - set(df.columns))}")
        continue

    # ensure datetime for ordering
    if not pd.api.types.is_datetime64_any_dtype(df["game_date"]):
        df["game_date"] = pd.to_datetime(df["game_date"], errors="coerce")

    # aggregate to batter-game level
    bg = (
        df.groupby(["batter_name", "game_date"], as_index=False)["home_run"]
          .sum()
          .assign(hit_hr=lambda t: (t["home_run"] > 0).astype(int))
          .sort_values(["batter_name", "game_date"])
    )

    g = bg.groupby("batter_name", group_keys=False)["hit_hr"]

    # previous game (1)
    bg["prev_game_hr"] = g.shift(1).fillna(0).astype(int)

    # previous 2 and 5 games (binary indicators)
    for N in (2, 5):
        col_name = f"prev_{N}g_hr"
        rolling_sum = g.apply(lambda s: s.shift(1).rolling(N, min_periods=1).sum())
        bg[col_name] = (rolling_sum > 0).astype(int)

    # merge back to at-bat level
    feats = ["batter_name", "game_date", "prev_game_hr", "prev_2g_hr", "prev_5g_hr"]
    globals()[name] = df.merge(bg[feats], on=["batter_name", "game_date"], how="left")

    print(f"{name}: added prev_game_hr, prev_2g_hr, prev_5g_hr")

at_bats_2021: added prev_game_hr, prev_2g_hr, prev_5g_hr
at_bats_2022: added prev_game_hr, prev_2g_hr, prev_5g_hr
at_bats_2023: added prev_game_hr, prev_2g_hr, prev_5g_hr
at_bats_2024: added prev_game_hr, prev_2g_hr, prev_5g_hr
at_bats_2025: added prev_game_hr, prev_2g_hr, prev_5g_hr


In [28]:
display(HTML("<h4>Season 2021</h4>")); display(at_bats_2021.head(5))
display(HTML("<h4>Season 2022</h4>")); display(at_bats_2022.head(5))
display(HTML("<h4>Season 2023</h4>")); display(at_bats_2023.head(5))
display(HTML("<h4>Season 2024</h4>")); display(at_bats_2024.head(5))
display(HTML("<h4>Season 2025</h4>")); display(at_bats_2025.head(5))


,game_date,pitcher_name,home_team,away_team,inning,inning_topbot,at_bat_number,batter_name,batter_id,pitch_number,...,rolling_batter_barrel_rate_40,rolling_batter_barrel_rate_20,rolling_batter_barrel_rate_10,rolling_pitcher_barrel_rate_40,rolling_pitcher_barrel_rate_20,rolling_pitcher_barrel_rate_10,home_run,prev_game_hr,prev_2g_hr,prev_5g_hr
0,2021-10-02,"Gsellman, Robert",ATL,NYM,6,Bot,51,A. j. Minter,621345,5,...,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0
1,2021-08-14,"Keller, Mitch",PIT,MIL,3,Top,19,Aaron Ashby,676879,4,...,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0
2,2021-08-14,"Keller, Mitch",PIT,MIL,4,Top,31,Aaron Ashby,676879,3,...,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0
3,2021-09-03,"Wainwright, Adam",MIL,STL,3,Bot,28,Aaron Ashby,676879,1,...,0.0,0.0,0.0,NaN,NaN,NaN,0,0,0,0
4,2021-09-03,"Wainwright, Adam",MIL,STL,6,Bot,46,Aaron Ashby,676879,7,...,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0


,game_date,pitcher_name,home_team,away_team,inning,inning_topbot,at_bat_number,batter_name,batter_id,pitch_number,...,rolling_batter_barrel_rate_40,rolling_batter_barrel_rate_20,rolling_batter_barrel_rate_10,rolling_pitcher_barrel_rate_40,rolling_pitcher_barrel_rate_20,rolling_pitcher_barrel_rate_10,home_run,prev_game_hr,prev_2g_hr,prev_5g_hr
0,2022-04-08,"Eovaldi, Nathan",NYY,BOS,2,Bot,17,Aaron Hicks,543305,8,...,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0
1,2022-04-08,"Eovaldi, Nathan",NYY,BOS,4,Bot,33,Aaron Hicks,543305,4,...,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0
2,2022-04-08,"Whitlock, Garrett",NYY,BOS,6,Bot,52,Aaron Hicks,543305,4,...,0.0,0.0,0.0,NaN,NaN,NaN,0,0,0,0
3,2022-04-08,"Strahm, Matt",NYY,BOS,8,Bot,67,Aaron Hicks,543305,6,...,0.0,0.0,0.0,NaN,NaN,NaN,0,0,0,0
4,2022-04-08,"Diekman, Jake",NYY,BOS,10,Bot,83,Aaron Hicks,543305,4,...,0.0,0.0,0.0,NaN,NaN,NaN,0,0,0,0


,game_date,pitcher_name,home_team,away_team,inning,inning_topbot,at_bat_number,batter_name,batter_id,pitch_number,...,rolling_batter_barrel_rate_40,rolling_batter_barrel_rate_20,rolling_batter_barrel_rate_10,rolling_pitcher_barrel_rate_40,rolling_pitcher_barrel_rate_20,rolling_pitcher_barrel_rate_10,home_run,prev_game_hr,prev_2g_hr,prev_5g_hr
0,2023-04-01,"Doval, Camilo",NYY,SF,9,Bot,79,Aaron Hicks,543305,7,...,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0
1,2023-04-03,"Walker, Taijuan",NYY,PHI,1,Bot,10,Aaron Hicks,543305,6,...,0.0,0.0,0.0,NaN,NaN,NaN,0,0,0,0
2,2023-04-03,"Walker, Taijuan",NYY,PHI,4,Bot,33,Aaron Hicks,543305,4,...,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0
3,2023-04-03,"Marte, Yunior",NYY,PHI,5,Bot,46,Aaron Hicks,543305,4,...,0.0,0.0,0.0,NaN,NaN,NaN,0,0,0,0
4,2023-04-03,"Vasquez, Andrew",NYY,PHI,7,Bot,64,Aaron Hicks,543305,2,...,0.0,0.0,0.0,NaN,NaN,NaN,0,0,0,0


,game_date,pitcher_name,home_team,away_team,inning,inning_topbot,at_bat_number,batter_name,batter_id,pitch_number,...,rolling_batter_barrel_rate_40,rolling_batter_barrel_rate_20,rolling_batter_barrel_rate_10,rolling_pitcher_barrel_rate_40,rolling_pitcher_barrel_rate_20,rolling_pitcher_barrel_rate_10,home_run,prev_game_hr,prev_2g_hr,prev_5g_hr
0,2024-03-20,"Hicks, Jordan",LAA,SF,1,Bot,6,Aaron Hicks,543305,1,...,NaN,NaN,NaN,NaN,NaN,NaN,0,0.0,0.0,0.0
1,2024-03-20,"Hicks, Jordan",LAA,SF,3,Bot,22,Aaron Hicks,543305,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0
2,2024-03-20,"Frisbee, Matt",LAA,SF,5,Bot,40,Aaron Hicks,543305,3,...,0.0,0.0,0.0,NaN,NaN,NaN,0,0.0,0.0,0.0
3,2024-03-22,"Cannon, Jonathan",LAA,CWS,1,Bot,5,Aaron Hicks,543305,1,...,0.0,0.0,0.0,NaN,NaN,NaN,0,0.0,0.0,0.0
4,2024-03-22,"Cannon, Jonathan",LAA,CWS,3,Bot,21,Aaron Hicks,543305,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0


,game_date,pitcher_name,home_team,away_team,inning,inning_topbot,at_bat_number,batter_name,batter_id,pitch_number,...,rolling_batter_barrel_rate_40,rolling_batter_barrel_rate_20,rolling_batter_barrel_rate_10,rolling_pitcher_barrel_rate_40,rolling_pitcher_barrel_rate_20,rolling_pitcher_barrel_rate_10,home_run,prev_game_hr,prev_2g_hr,prev_5g_hr
0,2025-03-18,"Buehler, Walker",NYY,BOS,1,Bot,6,Aaron Judge,592450,4,...,NaN,NaN,NaN,NaN,NaN,NaN,0,0.0,0.0,0.0
1,2025-03-18,"Buehler, Walker",NYY,BOS,4,Bot,25,Aaron Judge,592450,3,...,0.000000,0.000000,0.000000,0.0,0.0,0.0,0,0.0,0.0,0.0
2,2025-03-18,"Buehler, Walker",NYY,BOS,5,Bot,37,Aaron Judge,592450,4,...,0.500000,0.500000,0.500000,0.5,0.5,0.5,0,0.0,0.0,0.0
3,2025-03-19,"Schwellenbach, Spencer",NYY,ATL,1,Bot,6,Aaron Judge,592450,4,...,0.333333,0.333333,0.333333,NaN,NaN,NaN,0,0.0,0.0,0.0
4,2025-03-19,"Schwellenbach, Spencer",NYY,ATL,4,Bot,25,Aaron Judge,592450,3,...,0.250000,0.250000,0.250000,0.0,0.0,0.0,0,0.0,0.0,0.0


## Missing Data Imputation

There are several sources of missing data. Since we are using rolling rates batter and pitcher barrel rates, then naturally the first at-bat for both pitcerh and batter will have missing data. 

Second, since some batters and pitchers only appear once, then they will also have `NaN` for the missing data. 

**Solution**

For the first source, I impute the batters' and pitchers' mean barrel rates and 

While deleting that data may seem like a good solution, it will also get rid of the pitcher's data. 



In [29]:
'''
Move to a preprocessing script
'''

def compute_group_mean_with_overall(
    df: pd.DataFrame,
    group_col: str,
    value_col: str = "barrel",
    overall_id: str = "overall",
    mean_col_name: str | None = None
) -> pd.DataFrame:
    """
    Compute per-group mean of `value_col` plus a first row for the true overall mean.

    Parameters
    ----------
    df : pd.DataFrame
        Input data.
    group_col : str
        Column to group by (e.g., 'batter_id', 'pitcher_name').
    value_col : str, default 'barrel'
        Column whose mean is computed.
    overall_id : str, default 'overall'
        Label used for the overall row in `group_col`.
    mean_col_name : str | None
        Optional name of the output mean column; if None, uses f"{value_col}_mean".

    Returns
    -------
    pd.DataFrame
        DataFrame with an overall row first, followed by per-group means.
    """
    if mean_col_name is None:
        mean_col_name = f"{value_col}_mean"

    # Per-group mean
    per_group = (
        df.groupby(group_col, dropna=False)[value_col]
          .mean()
          .reset_index()
          .rename(columns={value_col: mean_col_name})
    )

    # True overall mean across all rows (not mean of means)
    overall_mean = df[value_col].mean()
    overall_row = pd.DataFrame({group_col: [overall_id], mean_col_name: [overall_mean]})

    # Combine with overall first
    out = pd.concat([overall_row, per_group], ignore_index=True)
    return out

### Batter Barrels Mean

This code creates a season-specific table of **average barrel rates by batter** for each year (2021–2025). For each `at_bats_YYYY` dataset, it computes the mean of the binary `barrel` indicator for each `batter_id` (i.e., that batter’s barrel rate) and **also includes the overall mean barrel rate** for the full season as a reference point. The resulting batter-level summary is saved as `batter_barrel_means_YYYY` with a clearly labeled column (e.g., `batter_mean_barrel_rate_2023`).


In [30]:
for y in range(2021, 2026):
    at_name  = f"at_bats_{y}"
    out_name = f"batter_barrel_means_{y}"

    df = globals().get(at_name)
    if df is None:
        print(f"{at_name}: (not loaded)")
        continue

    globals()[out_name] = compute_group_mean_with_overall(
        df,
        group_col="batter_id",
        value_col="barrel",
        mean_col_name=f"batter_mean_barrel_rate_{y}",
    )
    print(f"{out_name}: created")


batter_barrel_means_2021: created
batter_barrel_means_2022: created
batter_barrel_means_2023: created
batter_barrel_means_2024: created
batter_barrel_means_2025: created


In [31]:
display(HTML("<h4>Season 2021</h4>")); display(batter_barrel_means_2021.head(5))
display(HTML("<h4>Season 2022</h4>")); display(batter_barrel_means_2022.head(5))
display(HTML("<h4>Season 2023</h4>")); display(batter_barrel_means_2023.head(5))
display(HTML("<h4>Season 2024</h4>")); display(batter_barrel_means_2024.head(5))
display(HTML("<h4>Season 2025</h4>")); display(batter_barrel_means_2025.head(5))


,batter_id,batter_mean_barrel_rate_2021
0,overall,0.052941
1,405395,0.070707
2,408234,0.057034
3,425772,0.000000
4,425784,0.051282


,batter_id,batter_mean_barrel_rate_2022
0,overall,0.051166
1,405395,0.088319
2,408234,0.032258
3,425877,0.025735
4,429664,0.028846


,batter_id,batter_mean_barrel_rate_2023
0,overall,0.054427
1,408234,0.029730
2,425794,0.000000
3,443558,0.059211
4,444482,0.037825


,batter_id,batter_mean_barrel_rate_2024
0,overall,0.052176
1,444482,0.036765
2,453568,0.040777
3,455117,0.044872
4,456781,0.025890


,batter_id,batter_mean_barrel_rate_2025
0,overall,0.057851
1,455117,0.052632
2,456781,0.030928
3,457705,0.053097
4,457759,0.025510


### Pticher Barrels Mean

Similar to batter means, this code creates a season-specific table of **average barrel rates allowed by each pitcher** for each year (2021–2025). For each `at_bats_YYYY` dataset, it computes the mean of the binary `barrel` indicator within each `pitcher_name` group (interpreted as the pitcher’s barrel rate allowed) and **also includes the overall mean barrel rate allowed** for the full season as a reference. The resulting pitcher-level summary is saved as `pitcher_barrel_means_YYYY` with a clearly labeled column (e.g., `pitcher_mean_barrel_rate_allowed_2023`).


In [32]:
for y in range(2021, 2026):
    at_name  = f"at_bats_{y}"
    out_name = f"pitcher_barrel_means_{y}"

    df = globals().get(at_name)
    if df is None:
        print(f"{at_name}: (not loaded)")
        continue

    globals()[out_name] = compute_group_mean_with_overall(
        df,
        group_col="pitcher_name",
        value_col="barrel",
        mean_col_name=f"pitcher_mean_barrel_rate_allowed_{y}",
    )
    print(f"{out_name}: created")


pitcher_barrel_means_2021: created
pitcher_barrel_means_2022: created
pitcher_barrel_means_2023: created
pitcher_barrel_means_2024: created
pitcher_barrel_means_2025: created


In [33]:
display(HTML("<h4>Season 2021</h4>")); display(pitcher_barrel_means_2021.head(5))
display(HTML("<h4>Season 2022</h4>")); display(pitcher_barrel_means_2022.head(5))
display(HTML("<h4>Season 2023</h4>")); display(pitcher_barrel_means_2023.head(5))
display(HTML("<h4>Season 2024</h4>")); display(pitcher_barrel_means_2024.head(5))
display(HTML("<h4>Season 2025</h4>")); display(pitcher_barrel_means_2025.head(5))


,pitcher_name,pitcher_mean_barrel_rate_allowed_2021
0,overall,0.052941
1,"Abad, Fernando",0.036585
2,"Abbott, Cory",0.073171
3,"Abreu, Albert",0.051282
4,"Abreu, Bryan",0.043478


,pitcher_name,pitcher_mean_barrel_rate_allowed_2022
0,overall,0.051166
1,"Abbott, Cory",0.078704
2,"Abreu, Albert",0.034884
3,"Abreu, Bryan",0.020080
4,"Acevedo, Domingo",0.052632


,pitcher_name,pitcher_mean_barrel_rate_allowed_2023
0,overall,0.054427
1,"Abad, Fernando",0.093750
2,"Abbott, Andrew",0.058824
3,"Abbott, Cory",0.076087
4,"Abreu, Albert",0.044776


,pitcher_name,pitcher_mean_barrel_rate_allowed_2024
0,overall,0.052176
1,"Abbott, Andrew",0.066553
2,"Abney, Alaska",0.000000
3,"Abreu, Bryan",0.048485
4,"Adam, Jason",0.036913


,pitcher_name,pitcher_mean_barrel_rate_allowed_2025
0,overall,0.057851
1,"Abbott, Andrew",0.055954
2,"Abel, Mick",0.057471
3,"Abner, Philip",0.052632
4,"Abreu, Bryan",0.043189


### Imputing Missing Rolling Barrel Rates Using Prior-Season Means (Batters and Pitchers)

To handle missing values in rolling barrel-rate predictors—especially early in a season when a player has limited recent history—I impute these features using **prior-season group means**. For each season from **2022–2025**, I merge in the **previous year’s** batter-level and pitcher-level mean barrel rates, and then fill missing rolling rates accordingly. Missing rolling values are imputed **first** with the player’s prior-season mean; if that is unavailable, they fall back to the **overall mean** for the dataset.

This procedure is applied for rolling windows of **10, 20, and 40** at-bats for both batters (`rolling_batter_barrel_rate_{10,20,40}` using `batter_barrel_means_{y-1}`) and pitchers (`rolling_pitcher_barrel_rate_{10,20,40}` using `pitcher_barrel_means_{y-1}`). If a rolling feature is not already present, it is computed on the fly using only **prior at-bats** (via `shift(1)` to avoid look-ahead). The updated season DataFrames are then saved back for downstream modeling.


In [34]:
def impute_rolling_rates(
    df: pd.DataFrame,
    rolling_col: str,
    group_col: str,
    means_df: pd.DataFrame,
    mean_col: str,
    sort_first_by: str | None = None,
) -> pd.DataFrame:
    """
    Impute missing rolling rates using previous-season group means.
    If no group mean is available, fall back to the overall mean.
    Optionally sort by a chosen column (e.g., 'batter_name' or 'pitcher_name')
    plus standard game-context keys to ensure correct ordering before imputation.
    """
    if sort_first_by is not None:
        # Only include columns that exist to avoid KeyError
        ctx = [c for c in ['game_date', 'inning', 'at_bat_number', 'pitch_number', 'outs_when_up'] if c in df.columns]
        sort_cols = [sort_first_by] + ctx
        df = df.sort_values(sort_cols, kind="mergesort").reset_index(drop=True)

    # Merge group means onto current season data
    df = df.merge(means_df, how="left", on=group_col)

    # Calculate overall mean (true mean of rolling_col, ignoring NaNs)
    overall_mean = df[rolling_col].mean(skipna=True)

    # Fill missing rolling rates: first with group mean, then with overall mean
    df[rolling_col] = df[rolling_col].fillna(df[mean_col]).fillna(overall_mean)

    # Drop the temporary mean column
    df = df.drop(columns=[mean_col])

    return df

In [35]:
for y in range(2022, 2026):  # use previous-year means; skip 2021
    at_name    = f"at_bats_{y}"
    means_name = f"batter_barrel_means_{y-1}"
    mean_col   = f"batter_mean_barrel_rate_{y-1}"

    df = globals().get(at_name)
    means_df = globals().get(means_name)

    if df is None:
        print(f"{at_name}: (not loaded)")
        continue
    if means_df is None:
        print(f"{means_name}: (not loaded) — skipping {at_name}")
        continue

    # Ensure game_date is datetime if we need to compute rollings
    if not pd.api.types.is_datetime64_any_dtype(df.get("game_date", pd.Series(dtype="datetime64[ns]"))):
        if "game_date" in df.columns:
            df["game_date"] = pd.to_datetime(df["game_date"], errors="coerce")

    # Impute for each rolling window
    for window in (10, 20, 40):
        roll_col = f"rolling_batter_barrel_rate_{window}"

        # Compute the rolling column if missing
        if roll_col not in df.columns:
            if not {"batter_id", "barrel"}.issubset(df.columns):
                print(f"{at_name}: missing columns to compute {roll_col}; skipped")
                continue
            df[roll_col] = (
                df.groupby("batter_id")["barrel"]
                  .transform(lambda s: s.shift(1).rolling(window, min_periods=1).mean())
            )

        # Drop existing mean_col to avoid _x/_y suffixes inside impute()
        if mean_col in df.columns:
            df = df.drop(columns=[mean_col])

        out = impute_rolling_rates(
            df,
            rolling_col=roll_col,
            group_col="batter_id",
            means_df=means_df,
            mean_col=mean_col,
            sort_first_by="batter_name",
        )

        # Support functions that return a new df vs mutate in place
        if out is not None:
            df = out

    # Rebind the updated frame
    globals()[at_name] = df
    print(f"{at_name}: imputed rolling rates for 10, 20, 40 using {means_name}.{mean_col}")


at_bats_2022: imputed rolling rates for 10, 20, 40 using batter_barrel_means_2021.batter_mean_barrel_rate_2021
at_bats_2023: imputed rolling rates for 10, 20, 40 using batter_barrel_means_2022.batter_mean_barrel_rate_2022
at_bats_2024: imputed rolling rates for 10, 20, 40 using batter_barrel_means_2023.batter_mean_barrel_rate_2023
at_bats_2025: imputed rolling rates for 10, 20, 40 using batter_barrel_means_2024.batter_mean_barrel_rate_2024


In [36]:
display(HTML("<h4>Season 2022</h4>")); display(at_bats_2022.head(5))
display(HTML("<h4>Season 2023</h4>")); display(at_bats_2023.head(5))
display(HTML("<h4>Season 2024</h4>")); display(at_bats_2024.head(5))
display(HTML("<h4>Season 2025</h4>")); display(at_bats_2025.head(5))


,game_date,pitcher_name,home_team,away_team,inning,inning_topbot,at_bat_number,batter_name,batter_id,pitch_number,...,rolling_batter_barrel_rate_40,rolling_batter_barrel_rate_20,rolling_batter_barrel_rate_10,rolling_pitcher_barrel_rate_40,rolling_pitcher_barrel_rate_20,rolling_pitcher_barrel_rate_10,home_run,prev_game_hr,prev_2g_hr,prev_5g_hr
0,2022-04-08,"Eovaldi, Nathan",NYY,BOS,2,Bot,17,Aaron Hicks,543305,8,...,0.062992,0.062992,0.062992,NaN,NaN,NaN,0,0,0,0
1,2022-04-08,"Eovaldi, Nathan",NYY,BOS,4,Bot,33,Aaron Hicks,543305,4,...,0.000000,0.000000,0.000000,0.0,0.0,0.0,0,0,0,0
2,2022-04-08,"Whitlock, Garrett",NYY,BOS,6,Bot,52,Aaron Hicks,543305,4,...,0.000000,0.000000,0.000000,NaN,NaN,NaN,0,0,0,0
3,2022-04-08,"Strahm, Matt",NYY,BOS,8,Bot,67,Aaron Hicks,543305,6,...,0.000000,0.000000,0.000000,NaN,NaN,NaN,0,0,0,0
4,2022-04-08,"Diekman, Jake",NYY,BOS,10,Bot,83,Aaron Hicks,543305,4,...,0.000000,0.000000,0.000000,NaN,NaN,NaN,0,0,0,0


,game_date,pitcher_name,home_team,away_team,inning,inning_topbot,at_bat_number,batter_name,batter_id,pitch_number,...,rolling_batter_barrel_rate_40,rolling_batter_barrel_rate_20,rolling_batter_barrel_rate_10,rolling_pitcher_barrel_rate_40,rolling_pitcher_barrel_rate_20,rolling_pitcher_barrel_rate_10,home_run,prev_game_hr,prev_2g_hr,prev_5g_hr
0,2023-04-01,"Doval, Camilo",NYY,SF,9,Bot,79,Aaron Hicks,543305,7,...,0.035242,0.035242,0.035242,NaN,NaN,NaN,0,0,0,0
1,2023-04-03,"Walker, Taijuan",NYY,PHI,1,Bot,10,Aaron Hicks,543305,6,...,0.000000,0.000000,0.000000,NaN,NaN,NaN,0,0,0,0
2,2023-04-03,"Walker, Taijuan",NYY,PHI,4,Bot,33,Aaron Hicks,543305,4,...,0.000000,0.000000,0.000000,0.0,0.0,0.0,0,0,0,0
3,2023-04-03,"Marte, Yunior",NYY,PHI,5,Bot,46,Aaron Hicks,543305,4,...,0.000000,0.000000,0.000000,NaN,NaN,NaN,0,0,0,0
4,2023-04-03,"Vasquez, Andrew",NYY,PHI,7,Bot,64,Aaron Hicks,543305,2,...,0.000000,0.000000,0.000000,NaN,NaN,NaN,0,0,0,0


,game_date,pitcher_name,home_team,away_team,inning,inning_topbot,at_bat_number,batter_name,batter_id,pitch_number,...,rolling_batter_barrel_rate_40,rolling_batter_barrel_rate_20,rolling_batter_barrel_rate_10,rolling_pitcher_barrel_rate_40,rolling_pitcher_barrel_rate_20,rolling_pitcher_barrel_rate_10,home_run,prev_game_hr,prev_2g_hr,prev_5g_hr
0,2024-03-20,"Hicks, Jordan",LAA,SF,1,Bot,6,Aaron Hicks,543305,1,...,0.025559,0.025559,0.025559,NaN,NaN,NaN,0,0.0,0.0,0.0
1,2024-03-20,"Hicks, Jordan",LAA,SF,3,Bot,22,Aaron Hicks,543305,1,...,0.000000,0.000000,0.000000,0.0,0.0,0.0,0,0.0,0.0,0.0
2,2024-03-20,"Frisbee, Matt",LAA,SF,5,Bot,40,Aaron Hicks,543305,3,...,0.000000,0.000000,0.000000,NaN,NaN,NaN,0,0.0,0.0,0.0
3,2024-03-22,"Cannon, Jonathan",LAA,CWS,1,Bot,5,Aaron Hicks,543305,1,...,0.000000,0.000000,0.000000,NaN,NaN,NaN,0,0.0,0.0,0.0
4,2024-03-22,"Cannon, Jonathan",LAA,CWS,3,Bot,21,Aaron Hicks,543305,1,...,0.000000,0.000000,0.000000,0.0,0.0,0.0,0,0.0,0.0,0.0


,game_date,pitcher_name,home_team,away_team,inning,inning_topbot,at_bat_number,batter_name,batter_id,pitch_number,...,rolling_batter_barrel_rate_40,rolling_batter_barrel_rate_20,rolling_batter_barrel_rate_10,rolling_pitcher_barrel_rate_40,rolling_pitcher_barrel_rate_20,rolling_pitcher_barrel_rate_10,home_run,prev_game_hr,prev_2g_hr,prev_5g_hr
0,2025-03-18,"Buehler, Walker",NYY,BOS,1,Bot,6,Aaron Judge,592450,4,...,0.148252,0.148252,0.148252,NaN,NaN,NaN,0,0.0,0.0,0.0
1,2025-03-18,"Buehler, Walker",NYY,BOS,4,Bot,25,Aaron Judge,592450,3,...,0.000000,0.000000,0.000000,0.0,0.0,0.0,0,0.0,0.0,0.0
2,2025-03-18,"Buehler, Walker",NYY,BOS,5,Bot,37,Aaron Judge,592450,4,...,0.500000,0.500000,0.500000,0.5,0.5,0.5,0,0.0,0.0,0.0
3,2025-03-19,"Schwellenbach, Spencer",NYY,ATL,1,Bot,6,Aaron Judge,592450,4,...,0.333333,0.333333,0.333333,NaN,NaN,NaN,0,0.0,0.0,0.0
4,2025-03-19,"Schwellenbach, Spencer",NYY,ATL,4,Bot,25,Aaron Judge,592450,3,...,0.250000,0.250000,0.250000,0.0,0.0,0.0,0,0.0,0.0,0.0


In [37]:
for y in range(2022, 2026):  # use previous-year pitcher means; skip 2021
    at_name    = f"at_bats_{y}"
    means_name = f"pitcher_barrel_means_{y-1}"
    mean_col   = f"pitcher_mean_barrel_rate_allowed_{y-1}"

    df = globals().get(at_name)
    means_df = globals().get(means_name)

    if df is None:
        print(f"{at_name}: (not loaded)")
        continue
    if means_df is None:
        print(f"{means_name}: (not loaded) — skipping {at_name}")
        continue

    # Ensure datetime + sort BEFORE rolling computation
    if "game_date" in df.columns and not pd.api.types.is_datetime64_any_dtype(df["game_date"]):
        df["game_date"] = pd.to_datetime(df["game_date"], errors="coerce")

    sort_cols = ["pitcher_name", "game_date", "inning_topbot", "inning", "at_bat_number"]
    available = [c for c in sort_cols if c in df.columns]
    if available:
        df = df.sort_values(by=available, kind="mergesort").reset_index(drop=True)

    # FORCE recompute pitcher rollings (prevents stale wrong columns)
    for window in (10, 20, 40):
        roll_col = f"rolling_pitcher_barrel_rate_{window}"
        if roll_col in df.columns:
            df = df.drop(columns=[roll_col])

    # Compute + impute for each rolling window
    for window in (10, 20, 40):
        roll_col = f"rolling_pitcher_barrel_rate_{window}"

        need = {"pitcher_name", "barrel"}
        if not need.issubset(df.columns):
            print(f"{at_name}: missing columns to compute {roll_col}; skipped")
            continue

        df[roll_col] = (
            df.groupby("pitcher_name")["barrel"]
              .transform(lambda s: s.shift(1).rolling(window, min_periods=1).mean())
        )

        if mean_col in df.columns:
            df = df.drop(columns=[mean_col])

        df = impute_rolling_rates(
            df,
            rolling_col=roll_col,
            group_col="pitcher_name",
            means_df=means_df,
            mean_col=mean_col,
            sort_first_by="pitcher_name",
        )

    globals()[at_name] = df
    print(f"{at_name}: recomputed + imputed pitcher rolling rates (10, 20, 40) using {means_name}.{mean_col}")


at_bats_2022: recomputed + imputed pitcher rolling rates (10, 20, 40) using pitcher_barrel_means_2021.pitcher_mean_barrel_rate_allowed_2021
at_bats_2023: recomputed + imputed pitcher rolling rates (10, 20, 40) using pitcher_barrel_means_2022.pitcher_mean_barrel_rate_allowed_2022
at_bats_2024: recomputed + imputed pitcher rolling rates (10, 20, 40) using pitcher_barrel_means_2023.pitcher_mean_barrel_rate_allowed_2023
at_bats_2025: recomputed + imputed pitcher rolling rates (10, 20, 40) using pitcher_barrel_means_2024.pitcher_mean_barrel_rate_allowed_2024


In [38]:
display(HTML("<h4>Season 2022</h4>")); display(at_bats_2022.head(5))
display(HTML("<h4>Season 2023</h4>")); display(at_bats_2023.head(5))
display(HTML("<h4>Season 2024</h4>")); display(at_bats_2024.head(5))
display(HTML("<h4>Season 2025</h4>")); display(at_bats_2025.head(5))


,game_date,pitcher_name,home_team,away_team,inning,inning_topbot,at_bat_number,batter_name,batter_id,pitch_number,...,rolling_batter_barrel_rate_40,rolling_batter_barrel_rate_20,rolling_batter_barrel_rate_10,home_run,prev_game_hr,prev_2g_hr,prev_5g_hr,rolling_pitcher_barrel_rate_10,rolling_pitcher_barrel_rate_20,rolling_pitcher_barrel_rate_40
0,2022-06-19,"Abbott, Cory",WSH,PHI,9,Top,75,Alec Bohm,664761,1,...,0.050,0.05,0.0,0,0,0,0,0.073171,0.073171,0.073171
1,2022-06-19,"Abbott, Cory",WSH,PHI,9,Top,76,Bryson Stott,681082,7,...,0.000,0.00,0.0,0,0,0,0,0.000000,0.000000,0.000000
2,2022-06-19,"Abbott, Cory",WSH,PHI,9,Top,77,Matt Vierling,663837,3,...,0.100,0.15,0.2,0,0,1,1,0.000000,0.000000,0.000000
3,2022-07-13,"Abbott, Cory",WSH,SEA,8,Top,61,Carlos Santana,467793,4,...,0.075,0.10,0.0,0,1,1,1,0.000000,0.000000,0.000000
4,2022-07-13,"Abbott, Cory",WSH,SEA,8,Top,62,Eugenio Suárez,553993,5,...,0.100,0.10,0.1,0,0,0,1,0.000000,0.000000,0.000000


,game_date,pitcher_name,home_team,away_team,inning,inning_topbot,at_bat_number,batter_name,batter_id,pitch_number,...,rolling_batter_barrel_rate_40,rolling_batter_barrel_rate_20,rolling_batter_barrel_rate_10,home_run,prev_game_hr,prev_2g_hr,prev_5g_hr,rolling_pitcher_barrel_rate_10,rolling_pitcher_barrel_rate_20,rolling_pitcher_barrel_rate_40
0,2023-05-17,"Abad, Fernando",COL,CIN,5,Top,36,Stuart Fairchild,656413,2,...,0.025,0.00,0.0,0,0,0,0,0.054375,0.054371,0.054318
1,2023-05-17,"Abad, Fernando",COL,CIN,5,Top,37,Kevin Newman,621028,2,...,0.025,0.05,0.1,0,0,0,0,0.000000,0.000000,0.000000
2,2023-05-17,"Abad, Fernando",COL,CIN,5,Top,38,Wil Myers,571976,4,...,0.025,0.05,0.0,0,0,0,0,0.000000,0.000000,0.000000
3,2023-05-19,"Abad, Fernando",TEX,COL,8,Bot,66,Robbie Grossman,543257,5,...,0.025,0.00,0.0,0,0,0,1,0.000000,0.000000,0.000000
4,2023-05-19,"Abad, Fernando",TEX,COL,8,Bot,67,Ezequiel Duran,677649,6,...,0.125,0.10,0.2,0,0,1,1,0.000000,0.000000,0.000000


,game_date,pitcher_name,home_team,away_team,inning,inning_topbot,at_bat_number,batter_name,batter_id,pitch_number,...,rolling_batter_barrel_rate_40,rolling_batter_barrel_rate_20,rolling_batter_barrel_rate_10,home_run,prev_game_hr,prev_2g_hr,prev_5g_hr,rolling_pitcher_barrel_rate_10,rolling_pitcher_barrel_rate_20,rolling_pitcher_barrel_rate_40
0,2024-04-01,"Abbott, Andrew",PHI,CIN,1,Bot,5,Kyle Schwarber,656941,4,...,0.050000,0.0500,0.1,0,1.0,1.0,1.0,0.058824,0.058824,0.058824
1,2024-04-01,"Abbott, Andrew",PHI,CIN,1,Bot,6,Trea Turner,607208,8,...,0.041667,0.0500,0.0,0,0.0,0.0,0.0,0.000000,0.000000,0.000000
2,2024-04-01,"Abbott, Andrew",PHI,CIN,1,Bot,7,Bryce Harper,547180,6,...,0.062500,0.0625,0.0,0,0.0,0.0,0.0,0.000000,0.000000,0.000000
3,2024-04-01,"Abbott, Andrew",PHI,CIN,1,Bot,8,J. t. Realmuto,592663,5,...,0.125000,0.1500,0.2,0,0.0,1.0,1.0,0.000000,0.000000,0.000000
4,2024-04-01,"Abbott, Andrew",PHI,CIN,1,Bot,9,Alec Bohm,664761,3,...,0.000000,0.0000,0.0,0,0.0,0.0,0.0,0.000000,0.000000,0.000000


,game_date,pitcher_name,home_team,away_team,inning,inning_topbot,at_bat_number,batter_name,batter_id,pitch_number,...,rolling_batter_barrel_rate_40,rolling_batter_barrel_rate_20,rolling_batter_barrel_rate_10,home_run,prev_game_hr,prev_2g_hr,prev_5g_hr,rolling_pitcher_barrel_rate_10,rolling_pitcher_barrel_rate_20,rolling_pitcher_barrel_rate_40
0,2025-03-22,"Abbott, Andrew",SD,CIN,1,Bot,4,Fernando Tatís,665487,1,...,0.0,0.0,0.0,0,0.0,0.0,0.0,0.066553,0.066553,0.066553
1,2025-03-22,"Abbott, Andrew",SD,CIN,1,Bot,5,Luis Arráez,650333,4,...,0.0,0.0,0.0,0,0.0,0.0,0.0,0.000000,0.000000,0.000000
2,2025-03-22,"Abbott, Andrew",SD,CIN,1,Bot,6,Manny Machado,592518,3,...,0.0,0.0,0.0,0,0.0,0.0,0.0,0.000000,0.000000,0.000000
3,2025-03-22,"Abbott, Andrew",SD,CIN,1,Bot,7,Jackson Merrill,701538,6,...,0.0,0.0,0.0,0,0.0,0.0,0.0,0.000000,0.000000,0.000000
4,2025-03-22,"Abbott, Andrew",SD,CIN,2,Bot,11,Xander Bogaerts,593428,5,...,0.0,0.0,0.0,0,0.0,0.0,0.0,0.000000,0.000000,0.000000


### Unique Game IDs

Now, we add a unique `game_id` to each row to support reliable joins—especially when matching to doubleheader data where the same teams can play multiple games on the same date. The `game_id` is constructed by combining the game date (formatted as `YYYYMMDD`), the matchup (`away_team@home_team`), and a cleaned version of the pitcher’s name (commas removed and spaces standardized). This produces a consistent identifier that distinguishes games on the same day and helps prevent accidental mismatches when merging datasets.


In [39]:
'''
Move to Script
'''

def add_game_id(df):
    df = df.copy()

    # YYYYMMDD
    df["game_date_temp"] = pd.to_datetime(df["game_date"]).dt.strftime("%Y%m%d")

    # clean pitcher name:
    # - remove commas
    # - collapse whitespace
    # - replace spaces with underscores
    df["pitcher_id"] = (
        df["pitcher_name"]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.strip()
        .str.replace(r"\s+", "_", regex=True)
    )

    df["game_id"] = (
        df["game_date_temp"]
        + "_"
        + df["away_team"].astype(str)
        + "@"
        + df["home_team"].astype(str)
        + "_"
        + df["pitcher_id"]
    )

    return df.drop(columns=["game_date_temp", "pitcher_id"])


In [40]:
at_bats_2022 = add_game_id(at_bats_2022)
at_bats_2023 = add_game_id(at_bats_2023)
at_bats_2024 = add_game_id(at_bats_2024)
at_bats_2025 = add_game_id(at_bats_2025)

In [41]:
display(HTML("<h4>Season 2022</h4>")); display(at_bats_2022.head(5))
display(HTML("<h4>Season 2023</h4>")); display(at_bats_2023.head(5))
display(HTML("<h4>Season 2024</h4>")); display(at_bats_2024.head(5))
display(HTML("<h4>Season 2025</h4>")); display(at_bats_2025.head(5))

,game_date,pitcher_name,home_team,away_team,inning,inning_topbot,at_bat_number,batter_name,batter_id,pitch_number,...,rolling_batter_barrel_rate_20,rolling_batter_barrel_rate_10,home_run,prev_game_hr,prev_2g_hr,prev_5g_hr,rolling_pitcher_barrel_rate_10,rolling_pitcher_barrel_rate_20,rolling_pitcher_barrel_rate_40,game_id
0,2022-06-19,"Abbott, Cory",WSH,PHI,9,Top,75,Alec Bohm,664761,1,...,0.05,0.0,0,0,0,0,0.073171,0.073171,0.073171,20220619_PHI@WSH_Abbott_Cory
1,2022-06-19,"Abbott, Cory",WSH,PHI,9,Top,76,Bryson Stott,681082,7,...,0.00,0.0,0,0,0,0,0.000000,0.000000,0.000000,20220619_PHI@WSH_Abbott_Cory
2,2022-06-19,"Abbott, Cory",WSH,PHI,9,Top,77,Matt Vierling,663837,3,...,0.15,0.2,0,0,1,1,0.000000,0.000000,0.000000,20220619_PHI@WSH_Abbott_Cory
3,2022-07-13,"Abbott, Cory",WSH,SEA,8,Top,61,Carlos Santana,467793,4,...,0.10,0.0,0,1,1,1,0.000000,0.000000,0.000000,20220713_SEA@WSH_Abbott_Cory
4,2022-07-13,"Abbott, Cory",WSH,SEA,8,Top,62,Eugenio Suárez,553993,5,...,0.10,0.1,0,0,0,1,0.000000,0.000000,0.000000,20220713_SEA@WSH_Abbott_Cory


,game_date,pitcher_name,home_team,away_team,inning,inning_topbot,at_bat_number,batter_name,batter_id,pitch_number,...,rolling_batter_barrel_rate_20,rolling_batter_barrel_rate_10,home_run,prev_game_hr,prev_2g_hr,prev_5g_hr,rolling_pitcher_barrel_rate_10,rolling_pitcher_barrel_rate_20,rolling_pitcher_barrel_rate_40,game_id
0,2023-05-17,"Abad, Fernando",COL,CIN,5,Top,36,Stuart Fairchild,656413,2,...,0.00,0.0,0,0,0,0,0.054375,0.054371,0.054318,20230517_CIN@COL_Abad_Fernando
1,2023-05-17,"Abad, Fernando",COL,CIN,5,Top,37,Kevin Newman,621028,2,...,0.05,0.1,0,0,0,0,0.000000,0.000000,0.000000,20230517_CIN@COL_Abad_Fernando
2,2023-05-17,"Abad, Fernando",COL,CIN,5,Top,38,Wil Myers,571976,4,...,0.05,0.0,0,0,0,0,0.000000,0.000000,0.000000,20230517_CIN@COL_Abad_Fernando
3,2023-05-19,"Abad, Fernando",TEX,COL,8,Bot,66,Robbie Grossman,543257,5,...,0.00,0.0,0,0,0,1,0.000000,0.000000,0.000000,20230519_COL@TEX_Abad_Fernando
4,2023-05-19,"Abad, Fernando",TEX,COL,8,Bot,67,Ezequiel Duran,677649,6,...,0.10,0.2,0,0,1,1,0.000000,0.000000,0.000000,20230519_COL@TEX_Abad_Fernando


,game_date,pitcher_name,home_team,away_team,inning,inning_topbot,at_bat_number,batter_name,batter_id,pitch_number,...,rolling_batter_barrel_rate_20,rolling_batter_barrel_rate_10,home_run,prev_game_hr,prev_2g_hr,prev_5g_hr,rolling_pitcher_barrel_rate_10,rolling_pitcher_barrel_rate_20,rolling_pitcher_barrel_rate_40,game_id
0,2024-04-01,"Abbott, Andrew",PHI,CIN,1,Bot,5,Kyle Schwarber,656941,4,...,0.0500,0.1,0,1.0,1.0,1.0,0.058824,0.058824,0.058824,20240401_CIN@PHI_Abbott_Andrew
1,2024-04-01,"Abbott, Andrew",PHI,CIN,1,Bot,6,Trea Turner,607208,8,...,0.0500,0.0,0,0.0,0.0,0.0,0.000000,0.000000,0.000000,20240401_CIN@PHI_Abbott_Andrew
2,2024-04-01,"Abbott, Andrew",PHI,CIN,1,Bot,7,Bryce Harper,547180,6,...,0.0625,0.0,0,0.0,0.0,0.0,0.000000,0.000000,0.000000,20240401_CIN@PHI_Abbott_Andrew
3,2024-04-01,"Abbott, Andrew",PHI,CIN,1,Bot,8,J. t. Realmuto,592663,5,...,0.1500,0.2,0,0.0,1.0,1.0,0.000000,0.000000,0.000000,20240401_CIN@PHI_Abbott_Andrew
4,2024-04-01,"Abbott, Andrew",PHI,CIN,1,Bot,9,Alec Bohm,664761,3,...,0.0000,0.0,0,0.0,0.0,0.0,0.000000,0.000000,0.000000,20240401_CIN@PHI_Abbott_Andrew


,game_date,pitcher_name,home_team,away_team,inning,inning_topbot,at_bat_number,batter_name,batter_id,pitch_number,...,rolling_batter_barrel_rate_20,rolling_batter_barrel_rate_10,home_run,prev_game_hr,prev_2g_hr,prev_5g_hr,rolling_pitcher_barrel_rate_10,rolling_pitcher_barrel_rate_20,rolling_pitcher_barrel_rate_40,game_id
0,2025-03-22,"Abbott, Andrew",SD,CIN,1,Bot,4,Fernando Tatís,665487,1,...,0.0,0.0,0,0.0,0.0,0.0,0.066553,0.066553,0.066553,20250322_CIN@SD_Abbott_Andrew
1,2025-03-22,"Abbott, Andrew",SD,CIN,1,Bot,5,Luis Arráez,650333,4,...,0.0,0.0,0,0.0,0.0,0.0,0.000000,0.000000,0.000000,20250322_CIN@SD_Abbott_Andrew
2,2025-03-22,"Abbott, Andrew",SD,CIN,1,Bot,6,Manny Machado,592518,3,...,0.0,0.0,0,0.0,0.0,0.0,0.000000,0.000000,0.000000,20250322_CIN@SD_Abbott_Andrew
3,2025-03-22,"Abbott, Andrew",SD,CIN,1,Bot,7,Jackson Merrill,701538,6,...,0.0,0.0,0,0.0,0.0,0.0,0.000000,0.000000,0.000000,20250322_CIN@SD_Abbott_Andrew
4,2025-03-22,"Abbott, Andrew",SD,CIN,2,Bot,11,Xander Bogaerts,593428,5,...,0.0,0.0,0,0.0,0.0,0.0,0.000000,0.000000,0.000000,20250322_CIN@SD_Abbott_Andrew


## Double Headers

Next, we read in the doubleheader reference tables created in the [Double Headers](https://github.com/Sam-Gartenstein/home-run-prediction/blob/main/Double%20Headers.ipynb) notebook. These files identify games that were part of a doubleheader for each season, which is important because multiple games can share the same date and matchup, and we need a reliable way to distinguish them when merging datasets.

The code below loads `double_headers_{year}.csv` from `data/double_headers` for seasons 2022–2025, stores each season’s table in a dictionary, and also assigns convenience variables (`double_headers_22`, `double_headers_23`, etc.). Finally, it prints a small preview of each season’s doubleheader table for a quick sanity check.


In [42]:
indir = Path("data") / "double_headers"

double_headers = {}

for year in range(2022, 2026):
    fp = indir / f"double_headers_{year}.csv"
    if fp.exists():
        double_headers[year] = pd.read_csv(fp)
        print(f"[loaded] {fp}")
    else:
        print(f"[missing] {fp}")

# Recreate the same variable names if you want them back
double_headers_22 = double_headers.get(2022)
double_headers_23 = double_headers.get(2023)
double_headers_24 = double_headers.get(2024)
double_headers_25 = double_headers.get(2025)

display(HTML("<h4>Season 2022</h4>")); display(double_headers_22.head(10))
display(HTML("<h4>Season 2023</h4>")); display(double_headers_23.head(10))
display(HTML("<h4>Season 2024</h4>")); display(double_headers_24.head(10))
display(HTML("<h4>Season 2025</h4>")); display(double_headers_25.head(10))



[loaded] data/double_headers/double_headers_2022.csv
[loaded] data/double_headers/double_headers_2023.csv
[loaded] data/double_headers/double_headers_2024.csv
[loaded] data/double_headers/double_headers_2025.csv


,date,teams,away_team,home_team,game_number,bref_url,side,pitcher,game_id
0,2022-04-19,Arizona at Washington,AZ,WSH,1,https://www.baseball-reference.com/boxes/WAS/W...,away,"Bumgarner, Madison",20220419_AZ@WSH_Bumgarner_Madison
1,2022-04-19,Arizona at Washington,AZ,WSH,1,https://www.baseball-reference.com/boxes/WAS/W...,away,"Wendelken, J.B.",20220419_AZ@WSH_Wendelken_J.B.
2,2022-04-19,Arizona at Washington,AZ,WSH,1,https://www.baseball-reference.com/boxes/WAS/W...,away,"Pérez, Óliver",20220419_AZ@WSH_Pérez_Óliver
3,2022-04-19,Arizona at Washington,AZ,WSH,1,https://www.baseball-reference.com/boxes/WAS/W...,away,"Peacock, Matt",20220419_AZ@WSH_Peacock_Matt
4,2022-04-19,San Francisco at NY Mets,SF,NYM,1,https://www.baseball-reference.com/boxes/NYN/N...,away,"Cobb, Alex",20220419_SF@NYM_Cobb_Alex
5,2022-04-19,San Francisco at NY Mets,SF,NYM,1,https://www.baseball-reference.com/boxes/NYN/N...,away,"Leone, Dominic",20220419_SF@NYM_Leone_Dominic
6,2022-04-19,San Francisco at NY Mets,SF,NYM,1,https://www.baseball-reference.com/boxes/NYN/N...,away,"Álvarez, José",20220419_SF@NYM_Álvarez_José
7,2022-04-19,San Francisco at NY Mets,SF,NYM,1,https://www.baseball-reference.com/boxes/NYN/N...,away,"McGee, Jake",20220419_SF@NYM_McGee_Jake
8,2022-04-19,San Francisco at NY Mets,SF,NYM,1,https://www.baseball-reference.com/boxes/NYN/N...,away,"Rogers, Tyler",20220419_SF@NYM_Rogers_Tyler
9,2022-04-19,San Francisco at NY Mets,SF,NYM,1,https://www.baseball-reference.com/boxes/NYN/N...,away,"Doval, Camilo",20220419_SF@NYM_Doval_Camilo


,date,teams,away_team,home_team,game_number,bref_url,side,pitcher,game_id
0,2023-04-18,Cleveland at Detroit,CLE,DET,1,https://www.baseball-reference.com/boxes/DET/D...,away,"Gaddis, Hunter",20230418_CLE@DET_Gaddis_Hunter
1,2023-04-18,Cleveland at Detroit,CLE,DET,1,https://www.baseball-reference.com/boxes/DET/D...,away,"Morgan, Eli",20230418_CLE@DET_Morgan_Eli
2,2023-04-18,Cleveland at Detroit,CLE,DET,1,https://www.baseball-reference.com/boxes/DET/D...,away,"Sandlin, Nick",20230418_CLE@DET_Sandlin_Nick
3,2023-04-18,Cleveland at Detroit,CLE,DET,1,https://www.baseball-reference.com/boxes/DET/D...,away,"Karinchak, James",20230418_CLE@DET_Karinchak_James
4,2023-04-18,Philadelphia at Chicago Sox,PHI,CWS,1,https://www.baseball-reference.com/boxes/CHA/C...,away,"Wheeler, Zack",20230418_PHI@CWS_Wheeler_Zack
5,2023-04-18,Philadelphia at Chicago Sox,PHI,CWS,1,https://www.baseball-reference.com/boxes/CHA/C...,away,"Soto, Gregory",20230418_PHI@CWS_Soto_Gregory
6,2023-04-18,Philadelphia at Chicago Sox,PHI,CWS,1,https://www.baseball-reference.com/boxes/CHA/C...,away,"Kimbrel, Craig",20230418_PHI@CWS_Kimbrel_Craig
7,2023-04-18,Philadelphia at Chicago Sox,PHI,CWS,1,https://www.baseball-reference.com/boxes/CHA/C...,away,"Domínguez, Seranthony",20230418_PHI@CWS_Domínguez_Seranthony
8,2023-04-18,Philadelphia at Chicago Sox,PHI,CWS,1,https://www.baseball-reference.com/boxes/CHA/C...,away,"Alvarado, José",20230418_PHI@CWS_Alvarado_José
9,2023-04-18,Cleveland at Detroit,CLE,DET,1,https://www.baseball-reference.com/boxes/DET/D...,home,"Boyd, Matthew",20230418_CLE@DET_Boyd_Matthew


,date,teams,away_team,home_team,game_number,bref_url,side,pitcher,game_id
0,2024-04-04,Detroit at NY Mets,DET,NYM,1,https://www.baseball-reference.com/boxes/NYN/N...,away,"Mize, Casey",20240404_DET@NYM_Mize_Casey
1,2024-04-04,Detroit at NY Mets,DET,NYM,1,https://www.baseball-reference.com/boxes/NYN/N...,away,"Wentz, Joey",20240404_DET@NYM_Wentz_Joey
2,2024-04-04,Detroit at NY Mets,DET,NYM,1,https://www.baseball-reference.com/boxes/NYN/N...,away,"Lange, Alex",20240404_DET@NYM_Lange_Alex
3,2024-04-04,Detroit at NY Mets,DET,NYM,1,https://www.baseball-reference.com/boxes/NYN/N...,away,"Chafin, Andrew",20240404_DET@NYM_Chafin_Andrew
4,2024-04-04,Detroit at NY Mets,DET,NYM,1,https://www.baseball-reference.com/boxes/NYN/N...,away,"Foley, Jason",20240404_DET@NYM_Foley_Jason
5,2024-04-04,Detroit at NY Mets,DET,NYM,1,https://www.baseball-reference.com/boxes/NYN/N...,away,"Miller, Shelby",20240404_DET@NYM_Miller_Shelby
6,2024-04-04,Detroit at NY Mets,DET,NYM,1,https://www.baseball-reference.com/boxes/NYN/N...,home,"Houser, Adrian",20240404_DET@NYM_Houser_Adrian
7,2024-04-04,Detroit at NY Mets,DET,NYM,1,https://www.baseball-reference.com/boxes/NYN/N...,home,"Raley, Brooks",20240404_DET@NYM_Raley_Brooks
8,2024-04-04,Detroit at NY Mets,DET,NYM,1,https://www.baseball-reference.com/boxes/NYN/N...,home,"Smith, Drew",20240404_DET@NYM_Smith_Drew
9,2024-04-04,Detroit at NY Mets,DET,NYM,1,https://www.baseball-reference.com/boxes/NYN/N...,home,"Diekman, Jake",20240404_DET@NYM_Diekman_Jake


,date,teams,away_team,home_team,game_number,bref_url,side,pitcher,game_id
0,2025-04-06,St. Louis at Boston,STL,BOS,1,https://www.baseball-reference.com/boxes/BOS/B...,away,"Pallante, Andre",20250406_STL@BOS_Pallante_Andre
1,2025-04-06,St. Louis at Boston,STL,BOS,1,https://www.baseball-reference.com/boxes/BOS/B...,away,"Leahy, Kyle",20250406_STL@BOS_Leahy_Kyle
2,2025-04-06,St. Louis at Boston,STL,BOS,1,https://www.baseball-reference.com/boxes/BOS/B...,away,"Romero, JoJo",20250406_STL@BOS_Romero_JoJo
3,2025-04-06,St. Louis at Boston,STL,BOS,1,https://www.baseball-reference.com/boxes/BOS/B...,away,"Maton, Phil",20250406_STL@BOS_Maton_Phil
4,2025-04-06,St. Louis at Boston,STL,BOS,1,https://www.baseball-reference.com/boxes/BOS/B...,away,"Helsley, Ryan",20250406_STL@BOS_Helsley_Ryan
5,2025-04-06,St. Louis at Boston,STL,BOS,1,https://www.baseball-reference.com/boxes/BOS/B...,away,"Fernandez, Ryan",20250406_STL@BOS_Fernandez_Ryan
6,2025-04-06,St. Louis at Boston,STL,BOS,1,https://www.baseball-reference.com/boxes/BOS/B...,home,"Newcomb, Sean",20250406_STL@BOS_Newcomb_Sean
7,2025-04-06,St. Louis at Boston,STL,BOS,1,https://www.baseball-reference.com/boxes/BOS/B...,home,"Weissert, Greg",20250406_STL@BOS_Weissert_Greg
8,2025-04-06,St. Louis at Boston,STL,BOS,1,https://www.baseball-reference.com/boxes/BOS/B...,home,"Wilson, Justin",20250406_STL@BOS_Wilson_Justin
9,2025-04-06,St. Louis at Boston,STL,BOS,1,https://www.baseball-reference.com/boxes/BOS/B...,home,"Slaten, Justin",20250406_STL@BOS_Slaten_Justin


### Unique Headers

The code below gets the unique headers.

In [43]:
unique_ids_22 = double_headers_22["game_id"].unique()
unique_ids_23 = double_headers_23["game_id"].unique()
unique_ids_24 = double_headers_24["game_id"].unique()
unique_ids_25 = double_headers_25["game_id"].unique()

### Merging Doubleheader Game Numbers Into At-Bat Data

This function merges doubleheader metadata into the at-bat dataset by performing a **left join on `game_id`**. It first selects only the relevant fields from the doubleheader table—`game_id`, `game_number`, and `pitcher`—and removes duplicates to ensure a clean one-to-one mapping per game.

The merge then appends `game_number` (and the pitcher label from the doubleheader file) to `at_bats_df` while preserving all rows in the at-bat data. Games that are not part of a doubleheader will simply have missing (`NaN`) values for these added fields.


In [44]:
def merge_game_number_and_pitcher(at_bats_df: pd.DataFrame, double_headers_df: pd.DataFrame) -> pd.DataFrame:
    """
    Left-merge game_number + pitcher from double_headers_df into at_bats_df by game_id.
    Keeps only columns: game_id, game_number, pitcher from double_headers_df (deduped).
    """
    keep = double_headers_df[["game_id", "game_number", "pitcher"]].drop_duplicates()

    return at_bats_df.merge(keep, on="game_id", how="left")

In [45]:
at_bats_2022 = merge_game_number_and_pitcher(at_bats_2022, double_headers_22)
at_bats_2023 = merge_game_number_and_pitcher(at_bats_2023, double_headers_23)
at_bats_2024 = merge_game_number_and_pitcher(at_bats_2024, double_headers_24)
at_bats_2025 = merge_game_number_and_pitcher(at_bats_2025, double_headers_25)


In [46]:
display(HTML("<h4>Season 2022</h4>")); display(at_bats_2022.head(5))
display(HTML("<h4>Season 2023</h4>")); display(at_bats_2023.head(5))
display(HTML("<h4>Season 2024</h4>")); display(at_bats_2024.head(5))
display(HTML("<h4>Season 2025</h4>")); display(at_bats_2025.head(5))


,game_date,pitcher_name,home_team,away_team,inning,inning_topbot,at_bat_number,batter_name,batter_id,pitch_number,...,home_run,prev_game_hr,prev_2g_hr,prev_5g_hr,rolling_pitcher_barrel_rate_10,rolling_pitcher_barrel_rate_20,rolling_pitcher_barrel_rate_40,game_id,game_number,pitcher
0,2022-06-19,"Abbott, Cory",WSH,PHI,9,Top,75,Alec Bohm,664761,1,...,0,0,0,0,0.073171,0.073171,0.073171,20220619_PHI@WSH_Abbott_Cory,NaN,NaN
1,2022-06-19,"Abbott, Cory",WSH,PHI,9,Top,76,Bryson Stott,681082,7,...,0,0,0,0,0.000000,0.000000,0.000000,20220619_PHI@WSH_Abbott_Cory,NaN,NaN
2,2022-06-19,"Abbott, Cory",WSH,PHI,9,Top,77,Matt Vierling,663837,3,...,0,0,1,1,0.000000,0.000000,0.000000,20220619_PHI@WSH_Abbott_Cory,NaN,NaN
3,2022-07-13,"Abbott, Cory",WSH,SEA,8,Top,61,Carlos Santana,467793,4,...,0,1,1,1,0.000000,0.000000,0.000000,20220713_SEA@WSH_Abbott_Cory,1.0,"Abbott, Cory"
4,2022-07-13,"Abbott, Cory",WSH,SEA,8,Top,62,Eugenio Suárez,553993,5,...,0,0,0,1,0.000000,0.000000,0.000000,20220713_SEA@WSH_Abbott_Cory,1.0,"Abbott, Cory"


,game_date,pitcher_name,home_team,away_team,inning,inning_topbot,at_bat_number,batter_name,batter_id,pitch_number,...,home_run,prev_game_hr,prev_2g_hr,prev_5g_hr,rolling_pitcher_barrel_rate_10,rolling_pitcher_barrel_rate_20,rolling_pitcher_barrel_rate_40,game_id,game_number,pitcher
0,2023-05-17,"Abad, Fernando",COL,CIN,5,Top,36,Stuart Fairchild,656413,2,...,0,0,0,0,0.054375,0.054371,0.054318,20230517_CIN@COL_Abad_Fernando,NaN,NaN
1,2023-05-17,"Abad, Fernando",COL,CIN,5,Top,37,Kevin Newman,621028,2,...,0,0,0,0,0.000000,0.000000,0.000000,20230517_CIN@COL_Abad_Fernando,NaN,NaN
2,2023-05-17,"Abad, Fernando",COL,CIN,5,Top,38,Wil Myers,571976,4,...,0,0,0,0,0.000000,0.000000,0.000000,20230517_CIN@COL_Abad_Fernando,NaN,NaN
3,2023-05-19,"Abad, Fernando",TEX,COL,8,Bot,66,Robbie Grossman,543257,5,...,0,0,0,1,0.000000,0.000000,0.000000,20230519_COL@TEX_Abad_Fernando,NaN,NaN
4,2023-05-19,"Abad, Fernando",TEX,COL,8,Bot,67,Ezequiel Duran,677649,6,...,0,0,1,1,0.000000,0.000000,0.000000,20230519_COL@TEX_Abad_Fernando,NaN,NaN


,game_date,pitcher_name,home_team,away_team,inning,inning_topbot,at_bat_number,batter_name,batter_id,pitch_number,...,home_run,prev_game_hr,prev_2g_hr,prev_5g_hr,rolling_pitcher_barrel_rate_10,rolling_pitcher_barrel_rate_20,rolling_pitcher_barrel_rate_40,game_id,game_number,pitcher
0,2024-04-01,"Abbott, Andrew",PHI,CIN,1,Bot,5,Kyle Schwarber,656941,4,...,0,1.0,1.0,1.0,0.058824,0.058824,0.058824,20240401_CIN@PHI_Abbott_Andrew,NaN,NaN
1,2024-04-01,"Abbott, Andrew",PHI,CIN,1,Bot,6,Trea Turner,607208,8,...,0,0.0,0.0,0.0,0.000000,0.000000,0.000000,20240401_CIN@PHI_Abbott_Andrew,NaN,NaN
2,2024-04-01,"Abbott, Andrew",PHI,CIN,1,Bot,7,Bryce Harper,547180,6,...,0,0.0,0.0,0.0,0.000000,0.000000,0.000000,20240401_CIN@PHI_Abbott_Andrew,NaN,NaN
3,2024-04-01,"Abbott, Andrew",PHI,CIN,1,Bot,8,J. t. Realmuto,592663,5,...,0,0.0,1.0,1.0,0.000000,0.000000,0.000000,20240401_CIN@PHI_Abbott_Andrew,NaN,NaN
4,2024-04-01,"Abbott, Andrew",PHI,CIN,1,Bot,9,Alec Bohm,664761,3,...,0,0.0,0.0,0.0,0.000000,0.000000,0.000000,20240401_CIN@PHI_Abbott_Andrew,NaN,NaN


,game_date,pitcher_name,home_team,away_team,inning,inning_topbot,at_bat_number,batter_name,batter_id,pitch_number,...,home_run,prev_game_hr,prev_2g_hr,prev_5g_hr,rolling_pitcher_barrel_rate_10,rolling_pitcher_barrel_rate_20,rolling_pitcher_barrel_rate_40,game_id,game_number,pitcher
0,2025-03-22,"Abbott, Andrew",SD,CIN,1,Bot,4,Fernando Tatís,665487,1,...,0,0.0,0.0,0.0,0.066553,0.066553,0.066553,20250322_CIN@SD_Abbott_Andrew,NaN,NaN
1,2025-03-22,"Abbott, Andrew",SD,CIN,1,Bot,5,Luis Arráez,650333,4,...,0,0.0,0.0,0.0,0.000000,0.000000,0.000000,20250322_CIN@SD_Abbott_Andrew,NaN,NaN
2,2025-03-22,"Abbott, Andrew",SD,CIN,1,Bot,6,Manny Machado,592518,3,...,0,0.0,0.0,0.0,0.000000,0.000000,0.000000,20250322_CIN@SD_Abbott_Andrew,NaN,NaN
3,2025-03-22,"Abbott, Andrew",SD,CIN,1,Bot,7,Jackson Merrill,701538,6,...,0,0.0,0.0,0.0,0.000000,0.000000,0.000000,20250322_CIN@SD_Abbott_Andrew,NaN,NaN
4,2025-03-22,"Abbott, Andrew",SD,CIN,2,Bot,11,Xander Bogaerts,593428,5,...,0,0.0,0.0,0.0,0.000000,0.000000,0.000000,20250322_CIN@SD_Abbott_Andrew,NaN,NaN


###  Validating Doubleheader Pitcher Merges

To validate the doubleheader merge, we filter each season’s at-bat dataset down to the set of `game_id`s expected to appear in the doubleheader tables (`unique_ids_22`–`unique_ids_25`). We then sort the filtered rows by `game_number` and `game_id` to make the results easy to inspect.

Finally, we print the number of missing values in the `pitcher` column for each season. The goal is to confirm that, for these doubleheader games, the merged `pitcher` field is fully populated (i.e., there are no `NaN`/missing pitcher values).


In [47]:
'''
Move to Script
'''

def filter_and_sort_by_ids(
    df: pd.DataFrame,
    ids,
    id_col: str = "game_id",
    sort_cols: list[str] = None,
    ascending: list[bool] = None,
) -> pd.DataFrame:
    """
    Filter df to rows where df[id_col] is in ids, then sort and reset index.
    """
    if sort_cols is None:
        sort_cols = ["game_number", "game_id"]
    if ascending is None:
        ascending = [True] * len(sort_cols)

    out = df[df[id_col].isin(ids)].copy()
    out = out.sort_values(by=sort_cols, ascending=ascending).reset_index(drop=True)
    return out

In [48]:
at_bats_2022_test = filter_and_sort_by_ids(at_bats_2022, unique_ids_22)
at_bats_2023_test = filter_and_sort_by_ids(at_bats_2023, unique_ids_23)
at_bats_2024_test = filter_and_sort_by_ids(at_bats_2024, unique_ids_24)
at_bats_2025_test = filter_and_sort_by_ids(at_bats_2025, unique_ids_25)


In [49]:
print(at_bats_2022_test["pitcher"].isna().sum())
print(at_bats_2023_test["pitcher"].isna().sum())
print(at_bats_2024_test["pitcher"].isna().sum())
print(at_bats_2025_test["pitcher"].isna().sum())

0
0
0
0


#### Analysis

All pitchers were successfully matched with no missing values, confirming that the merge worked as intended and that a new identifier can now be created.


### Trimming `game_id` to a Game-Level Identifier

Now that the doubleheader merge is validated (i.e., no missing pitchers), we no longer need the pitcher component embedded in `game_id`. This function therefore trims `game_id` from a pitcher-specific identifier (e.g., `YYYYMMDD_AWAY@HOME_Pitcher_Name`) to a game-level identifier (`YYYYMMDD_AWAY@HOME`). It does this by splitting on underscores, keeping only the date and matchup portions, and overwriting the original `game_id` column with the trimmed version.


In [50]:
def trim_game_id_inplace(df: pd.DataFrame, col: str = "game_id") -> pd.DataFrame:
    """
    Overwrite game_id from:
    YYYYMMDD_AWAY@HOME_Pitcher_Name
    to:
    YYYYMMDD_AWAY@HOME
    """
    df = df.copy()
    df[col] = df[col].astype(str).str.split("_", n=2).str[:2].str.join("_")
    return df

In [51]:
at_bats_2022 = trim_game_id_inplace(at_bats_2022)
at_bats_2023 = trim_game_id_inplace(at_bats_2023)
at_bats_2024 = trim_game_id_inplace(at_bats_2024)
at_bats_2025 = trim_game_id_inplace(at_bats_2025)

display(HTML("<h4>Season 2022</h4>")); display(at_bats_2022.head(5))
display(HTML("<h4>Season 2023</h4>")); display(at_bats_2023.head(5))
display(HTML("<h4>Season 2024</h4>")); display(at_bats_2024.head(5))
display(HTML("<h4>Season 2025</h4>")); display(at_bats_2025.head(5))


,game_date,pitcher_name,home_team,away_team,inning,inning_topbot,at_bat_number,batter_name,batter_id,pitch_number,...,home_run,prev_game_hr,prev_2g_hr,prev_5g_hr,rolling_pitcher_barrel_rate_10,rolling_pitcher_barrel_rate_20,rolling_pitcher_barrel_rate_40,game_id,game_number,pitcher
0,2022-06-19,"Abbott, Cory",WSH,PHI,9,Top,75,Alec Bohm,664761,1,...,0,0,0,0,0.073171,0.073171,0.073171,20220619_PHI@WSH,NaN,NaN
1,2022-06-19,"Abbott, Cory",WSH,PHI,9,Top,76,Bryson Stott,681082,7,...,0,0,0,0,0.000000,0.000000,0.000000,20220619_PHI@WSH,NaN,NaN
2,2022-06-19,"Abbott, Cory",WSH,PHI,9,Top,77,Matt Vierling,663837,3,...,0,0,1,1,0.000000,0.000000,0.000000,20220619_PHI@WSH,NaN,NaN
3,2022-07-13,"Abbott, Cory",WSH,SEA,8,Top,61,Carlos Santana,467793,4,...,0,1,1,1,0.000000,0.000000,0.000000,20220713_SEA@WSH,1.0,"Abbott, Cory"
4,2022-07-13,"Abbott, Cory",WSH,SEA,8,Top,62,Eugenio Suárez,553993,5,...,0,0,0,1,0.000000,0.000000,0.000000,20220713_SEA@WSH,1.0,"Abbott, Cory"


,game_date,pitcher_name,home_team,away_team,inning,inning_topbot,at_bat_number,batter_name,batter_id,pitch_number,...,home_run,prev_game_hr,prev_2g_hr,prev_5g_hr,rolling_pitcher_barrel_rate_10,rolling_pitcher_barrel_rate_20,rolling_pitcher_barrel_rate_40,game_id,game_number,pitcher
0,2023-05-17,"Abad, Fernando",COL,CIN,5,Top,36,Stuart Fairchild,656413,2,...,0,0,0,0,0.054375,0.054371,0.054318,20230517_CIN@COL,NaN,NaN
1,2023-05-17,"Abad, Fernando",COL,CIN,5,Top,37,Kevin Newman,621028,2,...,0,0,0,0,0.000000,0.000000,0.000000,20230517_CIN@COL,NaN,NaN
2,2023-05-17,"Abad, Fernando",COL,CIN,5,Top,38,Wil Myers,571976,4,...,0,0,0,0,0.000000,0.000000,0.000000,20230517_CIN@COL,NaN,NaN
3,2023-05-19,"Abad, Fernando",TEX,COL,8,Bot,66,Robbie Grossman,543257,5,...,0,0,0,1,0.000000,0.000000,0.000000,20230519_COL@TEX,NaN,NaN
4,2023-05-19,"Abad, Fernando",TEX,COL,8,Bot,67,Ezequiel Duran,677649,6,...,0,0,1,1,0.000000,0.000000,0.000000,20230519_COL@TEX,NaN,NaN


,game_date,pitcher_name,home_team,away_team,inning,inning_topbot,at_bat_number,batter_name,batter_id,pitch_number,...,home_run,prev_game_hr,prev_2g_hr,prev_5g_hr,rolling_pitcher_barrel_rate_10,rolling_pitcher_barrel_rate_20,rolling_pitcher_barrel_rate_40,game_id,game_number,pitcher
0,2024-04-01,"Abbott, Andrew",PHI,CIN,1,Bot,5,Kyle Schwarber,656941,4,...,0,1.0,1.0,1.0,0.058824,0.058824,0.058824,20240401_CIN@PHI,NaN,NaN
1,2024-04-01,"Abbott, Andrew",PHI,CIN,1,Bot,6,Trea Turner,607208,8,...,0,0.0,0.0,0.0,0.000000,0.000000,0.000000,20240401_CIN@PHI,NaN,NaN
2,2024-04-01,"Abbott, Andrew",PHI,CIN,1,Bot,7,Bryce Harper,547180,6,...,0,0.0,0.0,0.0,0.000000,0.000000,0.000000,20240401_CIN@PHI,NaN,NaN
3,2024-04-01,"Abbott, Andrew",PHI,CIN,1,Bot,8,J. t. Realmuto,592663,5,...,0,0.0,1.0,1.0,0.000000,0.000000,0.000000,20240401_CIN@PHI,NaN,NaN
4,2024-04-01,"Abbott, Andrew",PHI,CIN,1,Bot,9,Alec Bohm,664761,3,...,0,0.0,0.0,0.0,0.000000,0.000000,0.000000,20240401_CIN@PHI,NaN,NaN


,game_date,pitcher_name,home_team,away_team,inning,inning_topbot,at_bat_number,batter_name,batter_id,pitch_number,...,home_run,prev_game_hr,prev_2g_hr,prev_5g_hr,rolling_pitcher_barrel_rate_10,rolling_pitcher_barrel_rate_20,rolling_pitcher_barrel_rate_40,game_id,game_number,pitcher
0,2025-03-22,"Abbott, Andrew",SD,CIN,1,Bot,4,Fernando Tatís,665487,1,...,0,0.0,0.0,0.0,0.066553,0.066553,0.066553,20250322_CIN@SD,NaN,NaN
1,2025-03-22,"Abbott, Andrew",SD,CIN,1,Bot,5,Luis Arráez,650333,4,...,0,0.0,0.0,0.0,0.000000,0.000000,0.000000,20250322_CIN@SD,NaN,NaN
2,2025-03-22,"Abbott, Andrew",SD,CIN,1,Bot,6,Manny Machado,592518,3,...,0,0.0,0.0,0.0,0.000000,0.000000,0.000000,20250322_CIN@SD,NaN,NaN
3,2025-03-22,"Abbott, Andrew",SD,CIN,1,Bot,7,Jackson Merrill,701538,6,...,0,0.0,0.0,0.0,0.000000,0.000000,0.000000,20250322_CIN@SD,NaN,NaN
4,2025-03-22,"Abbott, Andrew",SD,CIN,2,Bot,11,Xander Bogaerts,593428,5,...,0,0.0,0.0,0.0,0.000000,0.000000,0.000000,20250322_CIN@SD,NaN,NaN


### Appending Game Number to the ID

Now that we have a clean game-level `game_id` and have successfully merged in `game_number`, we append the game number to the identifier to clearly distinguish **Game 1 vs. Game 2** of a doubleheader. This prevents two games with the same date and matchup from sharing the same ID and ensures that subsequent merges and exploratory analysis treat them as separate events.

The function below updates `game_id` only when `game_number` is present (non-missing), converting identifiers like `YYYYMMDD_AWAY@HOME` into `YYYYMMDD_AWAY@HOME_1` (or `_2`). For non-doubleheader games where `game_number` is missing, `game_id` is left unchanged.


In [52]:
def append_game_number_to_game_id(df: pd.DataFrame,
                                  game_id_col: str = "game_id",
                                  game_number_col: str = "game_number") -> pd.DataFrame:
    """
    If game_number is not NaN, append it to game_id:
    YYYYMMDD_AWAY@HOME  ->  YYYYMMDD_AWAY@HOME_1

    Leaves game_id unchanged when game_number is NaN.
    """
    df = df.copy()

    mask = df[game_number_col].notna()

    df.loc[mask, game_id_col] = (
        df.loc[mask, game_id_col].astype(str)
        + "_"
        + df.loc[mask, game_number_col].astype(int).astype(str)
    )

    return df

In [53]:
at_bats_2022 = append_game_number_to_game_id(at_bats_2022)
at_bats_2023 = append_game_number_to_game_id(at_bats_2023)
at_bats_2024 = append_game_number_to_game_id(at_bats_2024)
at_bats_2025 = append_game_number_to_game_id(at_bats_2025)

display(HTML("<h4>Season 2022</h4>")); display(at_bats_2022.head(5))
display(HTML("<h4>Season 2023</h4>")); display(at_bats_2023.head(5))
display(HTML("<h4>Season 2024</h4>")); display(at_bats_2024.head(5))
display(HTML("<h4>Season 2025</h4>")); display(at_bats_2025.head(5))


,game_date,pitcher_name,home_team,away_team,inning,inning_topbot,at_bat_number,batter_name,batter_id,pitch_number,...,home_run,prev_game_hr,prev_2g_hr,prev_5g_hr,rolling_pitcher_barrel_rate_10,rolling_pitcher_barrel_rate_20,rolling_pitcher_barrel_rate_40,game_id,game_number,pitcher
0,2022-06-19,"Abbott, Cory",WSH,PHI,9,Top,75,Alec Bohm,664761,1,...,0,0,0,0,0.073171,0.073171,0.073171,20220619_PHI@WSH,NaN,NaN
1,2022-06-19,"Abbott, Cory",WSH,PHI,9,Top,76,Bryson Stott,681082,7,...,0,0,0,0,0.000000,0.000000,0.000000,20220619_PHI@WSH,NaN,NaN
2,2022-06-19,"Abbott, Cory",WSH,PHI,9,Top,77,Matt Vierling,663837,3,...,0,0,1,1,0.000000,0.000000,0.000000,20220619_PHI@WSH,NaN,NaN
3,2022-07-13,"Abbott, Cory",WSH,SEA,8,Top,61,Carlos Santana,467793,4,...,0,1,1,1,0.000000,0.000000,0.000000,20220713_SEA@WSH_1,1.0,"Abbott, Cory"
4,2022-07-13,"Abbott, Cory",WSH,SEA,8,Top,62,Eugenio Suárez,553993,5,...,0,0,0,1,0.000000,0.000000,0.000000,20220713_SEA@WSH_1,1.0,"Abbott, Cory"


,game_date,pitcher_name,home_team,away_team,inning,inning_topbot,at_bat_number,batter_name,batter_id,pitch_number,...,home_run,prev_game_hr,prev_2g_hr,prev_5g_hr,rolling_pitcher_barrel_rate_10,rolling_pitcher_barrel_rate_20,rolling_pitcher_barrel_rate_40,game_id,game_number,pitcher
0,2023-05-17,"Abad, Fernando",COL,CIN,5,Top,36,Stuart Fairchild,656413,2,...,0,0,0,0,0.054375,0.054371,0.054318,20230517_CIN@COL,NaN,NaN
1,2023-05-17,"Abad, Fernando",COL,CIN,5,Top,37,Kevin Newman,621028,2,...,0,0,0,0,0.000000,0.000000,0.000000,20230517_CIN@COL,NaN,NaN
2,2023-05-17,"Abad, Fernando",COL,CIN,5,Top,38,Wil Myers,571976,4,...,0,0,0,0,0.000000,0.000000,0.000000,20230517_CIN@COL,NaN,NaN
3,2023-05-19,"Abad, Fernando",TEX,COL,8,Bot,66,Robbie Grossman,543257,5,...,0,0,0,1,0.000000,0.000000,0.000000,20230519_COL@TEX,NaN,NaN
4,2023-05-19,"Abad, Fernando",TEX,COL,8,Bot,67,Ezequiel Duran,677649,6,...,0,0,1,1,0.000000,0.000000,0.000000,20230519_COL@TEX,NaN,NaN


,game_date,pitcher_name,home_team,away_team,inning,inning_topbot,at_bat_number,batter_name,batter_id,pitch_number,...,home_run,prev_game_hr,prev_2g_hr,prev_5g_hr,rolling_pitcher_barrel_rate_10,rolling_pitcher_barrel_rate_20,rolling_pitcher_barrel_rate_40,game_id,game_number,pitcher
0,2024-04-01,"Abbott, Andrew",PHI,CIN,1,Bot,5,Kyle Schwarber,656941,4,...,0,1.0,1.0,1.0,0.058824,0.058824,0.058824,20240401_CIN@PHI,NaN,NaN
1,2024-04-01,"Abbott, Andrew",PHI,CIN,1,Bot,6,Trea Turner,607208,8,...,0,0.0,0.0,0.0,0.000000,0.000000,0.000000,20240401_CIN@PHI,NaN,NaN
2,2024-04-01,"Abbott, Andrew",PHI,CIN,1,Bot,7,Bryce Harper,547180,6,...,0,0.0,0.0,0.0,0.000000,0.000000,0.000000,20240401_CIN@PHI,NaN,NaN
3,2024-04-01,"Abbott, Andrew",PHI,CIN,1,Bot,8,J. t. Realmuto,592663,5,...,0,0.0,1.0,1.0,0.000000,0.000000,0.000000,20240401_CIN@PHI,NaN,NaN
4,2024-04-01,"Abbott, Andrew",PHI,CIN,1,Bot,9,Alec Bohm,664761,3,...,0,0.0,0.0,0.0,0.000000,0.000000,0.000000,20240401_CIN@PHI,NaN,NaN


,game_date,pitcher_name,home_team,away_team,inning,inning_topbot,at_bat_number,batter_name,batter_id,pitch_number,...,home_run,prev_game_hr,prev_2g_hr,prev_5g_hr,rolling_pitcher_barrel_rate_10,rolling_pitcher_barrel_rate_20,rolling_pitcher_barrel_rate_40,game_id,game_number,pitcher
0,2025-03-22,"Abbott, Andrew",SD,CIN,1,Bot,4,Fernando Tatís,665487,1,...,0,0.0,0.0,0.0,0.066553,0.066553,0.066553,20250322_CIN@SD,NaN,NaN
1,2025-03-22,"Abbott, Andrew",SD,CIN,1,Bot,5,Luis Arráez,650333,4,...,0,0.0,0.0,0.0,0.000000,0.000000,0.000000,20250322_CIN@SD,NaN,NaN
2,2025-03-22,"Abbott, Andrew",SD,CIN,1,Bot,6,Manny Machado,592518,3,...,0,0.0,0.0,0.0,0.000000,0.000000,0.000000,20250322_CIN@SD,NaN,NaN
3,2025-03-22,"Abbott, Andrew",SD,CIN,1,Bot,7,Jackson Merrill,701538,6,...,0,0.0,0.0,0.0,0.000000,0.000000,0.000000,20250322_CIN@SD,NaN,NaN
4,2025-03-22,"Abbott, Andrew",SD,CIN,2,Bot,11,Xander Bogaerts,593428,5,...,0,0.0,0.0,0.0,0.000000,0.000000,0.000000,20250322_CIN@SD,NaN,NaN


### Dropping Game Number and Pitcher

Now, we can drop the `game_number` and `pitcher` columns.

In [54]:
def drop_game_number_and_pitcher(df: pd.DataFrame) -> pd.DataFrame:
    return df.drop(columns=["game_number", "pitcher"])

In [55]:
at_bats_2022 = drop_game_number_and_pitcher(at_bats_2022)
at_bats_2023 = drop_game_number_and_pitcher(at_bats_2023)
at_bats_2024 = drop_game_number_and_pitcher(at_bats_2024)
at_bats_2025 = drop_game_number_and_pitcher(at_bats_2025)

display(HTML("<h4>Season 2022</h4>")); display(at_bats_2022.head(5))
display(HTML("<h4>Season 2023</h4>")); display(at_bats_2023.head(5))
display(HTML("<h4>Season 2024</h4>")); display(at_bats_2024.head(5))
display(HTML("<h4>Season 2025</h4>")); display(at_bats_2025.head(5))


,game_date,pitcher_name,home_team,away_team,inning,inning_topbot,at_bat_number,batter_name,batter_id,pitch_number,...,rolling_batter_barrel_rate_20,rolling_batter_barrel_rate_10,home_run,prev_game_hr,prev_2g_hr,prev_5g_hr,rolling_pitcher_barrel_rate_10,rolling_pitcher_barrel_rate_20,rolling_pitcher_barrel_rate_40,game_id
0,2022-06-19,"Abbott, Cory",WSH,PHI,9,Top,75,Alec Bohm,664761,1,...,0.05,0.0,0,0,0,0,0.073171,0.073171,0.073171,20220619_PHI@WSH
1,2022-06-19,"Abbott, Cory",WSH,PHI,9,Top,76,Bryson Stott,681082,7,...,0.00,0.0,0,0,0,0,0.000000,0.000000,0.000000,20220619_PHI@WSH
2,2022-06-19,"Abbott, Cory",WSH,PHI,9,Top,77,Matt Vierling,663837,3,...,0.15,0.2,0,0,1,1,0.000000,0.000000,0.000000,20220619_PHI@WSH
3,2022-07-13,"Abbott, Cory",WSH,SEA,8,Top,61,Carlos Santana,467793,4,...,0.10,0.0,0,1,1,1,0.000000,0.000000,0.000000,20220713_SEA@WSH_1
4,2022-07-13,"Abbott, Cory",WSH,SEA,8,Top,62,Eugenio Suárez,553993,5,...,0.10,0.1,0,0,0,1,0.000000,0.000000,0.000000,20220713_SEA@WSH_1


,game_date,pitcher_name,home_team,away_team,inning,inning_topbot,at_bat_number,batter_name,batter_id,pitch_number,...,rolling_batter_barrel_rate_20,rolling_batter_barrel_rate_10,home_run,prev_game_hr,prev_2g_hr,prev_5g_hr,rolling_pitcher_barrel_rate_10,rolling_pitcher_barrel_rate_20,rolling_pitcher_barrel_rate_40,game_id
0,2023-05-17,"Abad, Fernando",COL,CIN,5,Top,36,Stuart Fairchild,656413,2,...,0.00,0.0,0,0,0,0,0.054375,0.054371,0.054318,20230517_CIN@COL
1,2023-05-17,"Abad, Fernando",COL,CIN,5,Top,37,Kevin Newman,621028,2,...,0.05,0.1,0,0,0,0,0.000000,0.000000,0.000000,20230517_CIN@COL
2,2023-05-17,"Abad, Fernando",COL,CIN,5,Top,38,Wil Myers,571976,4,...,0.05,0.0,0,0,0,0,0.000000,0.000000,0.000000,20230517_CIN@COL
3,2023-05-19,"Abad, Fernando",TEX,COL,8,Bot,66,Robbie Grossman,543257,5,...,0.00,0.0,0,0,0,1,0.000000,0.000000,0.000000,20230519_COL@TEX
4,2023-05-19,"Abad, Fernando",TEX,COL,8,Bot,67,Ezequiel Duran,677649,6,...,0.10,0.2,0,0,1,1,0.000000,0.000000,0.000000,20230519_COL@TEX


,game_date,pitcher_name,home_team,away_team,inning,inning_topbot,at_bat_number,batter_name,batter_id,pitch_number,...,rolling_batter_barrel_rate_20,rolling_batter_barrel_rate_10,home_run,prev_game_hr,prev_2g_hr,prev_5g_hr,rolling_pitcher_barrel_rate_10,rolling_pitcher_barrel_rate_20,rolling_pitcher_barrel_rate_40,game_id
0,2024-04-01,"Abbott, Andrew",PHI,CIN,1,Bot,5,Kyle Schwarber,656941,4,...,0.0500,0.1,0,1.0,1.0,1.0,0.058824,0.058824,0.058824,20240401_CIN@PHI
1,2024-04-01,"Abbott, Andrew",PHI,CIN,1,Bot,6,Trea Turner,607208,8,...,0.0500,0.0,0,0.0,0.0,0.0,0.000000,0.000000,0.000000,20240401_CIN@PHI
2,2024-04-01,"Abbott, Andrew",PHI,CIN,1,Bot,7,Bryce Harper,547180,6,...,0.0625,0.0,0,0.0,0.0,0.0,0.000000,0.000000,0.000000,20240401_CIN@PHI
3,2024-04-01,"Abbott, Andrew",PHI,CIN,1,Bot,8,J. t. Realmuto,592663,5,...,0.1500,0.2,0,0.0,1.0,1.0,0.000000,0.000000,0.000000,20240401_CIN@PHI
4,2024-04-01,"Abbott, Andrew",PHI,CIN,1,Bot,9,Alec Bohm,664761,3,...,0.0000,0.0,0,0.0,0.0,0.0,0.000000,0.000000,0.000000,20240401_CIN@PHI


,game_date,pitcher_name,home_team,away_team,inning,inning_topbot,at_bat_number,batter_name,batter_id,pitch_number,...,rolling_batter_barrel_rate_20,rolling_batter_barrel_rate_10,home_run,prev_game_hr,prev_2g_hr,prev_5g_hr,rolling_pitcher_barrel_rate_10,rolling_pitcher_barrel_rate_20,rolling_pitcher_barrel_rate_40,game_id
0,2025-03-22,"Abbott, Andrew",SD,CIN,1,Bot,4,Fernando Tatís,665487,1,...,0.0,0.0,0,0.0,0.0,0.0,0.066553,0.066553,0.066553,20250322_CIN@SD
1,2025-03-22,"Abbott, Andrew",SD,CIN,1,Bot,5,Luis Arráez,650333,4,...,0.0,0.0,0,0.0,0.0,0.0,0.000000,0.000000,0.000000,20250322_CIN@SD
2,2025-03-22,"Abbott, Andrew",SD,CIN,1,Bot,6,Manny Machado,592518,3,...,0.0,0.0,0,0.0,0.0,0.0,0.000000,0.000000,0.000000,20250322_CIN@SD
3,2025-03-22,"Abbott, Andrew",SD,CIN,1,Bot,7,Jackson Merrill,701538,6,...,0.0,0.0,0,0.0,0.0,0.0,0.000000,0.000000,0.000000,20250322_CIN@SD
4,2025-03-22,"Abbott, Andrew",SD,CIN,2,Bot,11,Xander Bogaerts,593428,5,...,0.0,0.0,0,0.0,0.0,0.0,0.000000,0.000000,0.000000,20250322_CIN@SD


## Exporting

With all identifiers and rolling features finalized, we now export the cleaned at-bat datasets for downstream exploratory data analysis (EDA) and Bayesian modeling. This export step ensures we are working from a consistent, reproducible version of the data moving forward.

In [56]:
out_dir = Path("data/at_bats")
out_dir.mkdir(parents=True, exist_ok=True)

for y in range(2022, 2026):
    var_name = f"at_bats_{y}"
    df = globals().get(var_name)
    if df is None:
        print(f"{var_name}: (not loaded)")
        continue

    out_path = out_dir / f"{var_name}.csv"
    df.to_csv(out_path, index=False)
    print(f"{var_name}: saved to {out_path}")


at_bats_2022: saved to data/at_bats/at_bats_2022.csv
at_bats_2023: saved to data/at_bats/at_bats_2023.csv
at_bats_2024: saved to data/at_bats/at_bats_2024.csv
at_bats_2025: saved to data/at_bats/at_bats_2025.csv
